In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install psutil thop

In [ ]:
import os, math, copy, random, time, threading, gc, itertools, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from thop import profile

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

In [ ]:
# =========================================================
# PATHS
# =========================================================

PROJECT_DIR    = "/content/drive/MyDrive/298/Elec/proj_v3"
BASE_MODEL_DIR = "/content/drive/MyDrive/298/Elec/proj_v3/baseModel"
DATASET_PATH   = "/content/drive/MyDrive/298/Elec/Data/LD2011_2014.txt"

FGSM_DIR = os.path.join(PROJECT_DIR, "fgsm")
BIM_DIR  = os.path.join(PROJECT_DIR, "bim")
PGD_DIR  = os.path.join(PROJECT_DIR, "pgd")

COMMON_ARTIFACTS_DIR = os.path.join(PROJECT_DIR, "common_artifacts")
COMMON_LOGS_DIR      = os.path.join(PROJECT_DIR, "common_logs")
COMP_METRICS_DIR     = os.path.join(PROJECT_DIR, "computational_metrics")

for d in [
    PROJECT_DIR, BASE_MODEL_DIR,
    FGSM_DIR, BIM_DIR, PGD_DIR,
    COMMON_ARTIFACTS_DIR, COMMON_LOGS_DIR, COMP_METRICS_DIR
]:
    os.makedirs(d, exist_ok=True)

# attack-specific directories
FGSM_STUDENTS_DIR      = os.path.join(FGSM_DIR, "students")
FGSM_STUDENTS_ADV_DIR  = os.path.join(FGSM_DIR, "students_adv")
FGSM_RESULTS_DIR       = os.path.join(FGSM_DIR, "results")
FGSM_LOGS_DIR          = os.path.join(FGSM_RESULTS_DIR, "train_logs")

BIM_STUDENTS_DIR       = os.path.join(BIM_DIR, "students")
BIM_STUDENTS_ADV_DIR   = os.path.join(BIM_DIR, "students_adv")
BIM_RESULTS_DIR        = os.path.join(BIM_DIR, "results")
BIM_LOGS_DIR           = os.path.join(BIM_RESULTS_DIR, "train_logs")

PGD_STUDENTS_DIR       = os.path.join(PGD_DIR, "students")
PGD_STUDENTS_ADV_DIR   = os.path.join(PGD_DIR, "students_adv")
PGD_RESULTS_DIR        = os.path.join(PGD_DIR, "results")
PGD_LOGS_DIR           = os.path.join(PGD_RESULTS_DIR, "train_logs")

for d in [
    FGSM_STUDENTS_DIR, FGSM_STUDENTS_ADV_DIR, FGSM_RESULTS_DIR, FGSM_LOGS_DIR,
    BIM_STUDENTS_DIR,  BIM_STUDENTS_ADV_DIR,  BIM_RESULTS_DIR,  BIM_LOGS_DIR,
    PGD_STUDENTS_DIR,  PGD_STUDENTS_ADV_DIR,  PGD_RESULTS_DIR,  PGD_LOGS_DIR,
]:
    os.makedirs(d, exist_ok=True)

# base outputs
BASE_CKPT_PATH         = os.path.join(BASE_MODEL_DIR, "elec_base_best.pth")
BASE_TRAIN_LOG_CSV     = os.path.join(COMMON_LOGS_DIR, "elec_base_train_log.csv")
BASE_RESOURCE_TRACE_CSV   = os.path.join(COMP_METRICS_DIR, "base_resource_trace.csv")
BASE_RESOURCE_SUMMARY_CSV = os.path.join(COMP_METRICS_DIR, "base_resource_summary.csv")
BASE_PROFILE_CSV          = os.path.join(COMP_METRICS_DIR, "base_model_profile.csv")

# clean eval
FGSM_CLEAN_EVAL_CSV = os.path.join(FGSM_RESULTS_DIR, "students_clean_eval.csv")
BIM_CLEAN_EVAL_CSV  = os.path.join(BIM_RESULTS_DIR,  "students_clean_eval.csv")
PGD_CLEAN_EVAL_CSV  = os.path.join(PGD_RESULTS_DIR,  "students_clean_eval.csv")

# train resource summary
FGSM_TRAIN_RESOURCE_CSV = os.path.join(COMP_METRICS_DIR, "fgsm_student_train_resource_summary.csv")
BIM_TRAIN_RESOURCE_CSV  = os.path.join(COMP_METRICS_DIR, "bim_student_train_resource_summary.csv")
PGD_TRAIN_RESOURCE_CSV  = os.path.join(COMP_METRICS_DIR, "pgd_student_train_resource_summary.csv")

# fgsm result files
FGSM_SINGLE_RUN_CSV        = os.path.join(FGSM_RESULTS_DIR, "fgsm_morphence_single_run.csv")
FGSM_STATS_CSV             = os.path.join(FGSM_RESULTS_DIR, "fgsm_morphence_single_pass_stats.csv")
FGSM_WEIGHT_DIVERSITY_CSV  = os.path.join(FGSM_RESULTS_DIR, "fgsm_weight_diversity.csv")
FGSM_PRED_DIVERSITY_CSV    = os.path.join(FGSM_RESULTS_DIR, "fgsm_prediction_diversity.csv")
FGSM_XFER_MATRIX_CSV       = os.path.join(FGSM_RESULTS_DIR, "fgsm_transfer_matrix.csv")
FGSM_EVAL_TRACE_CSV        = os.path.join(COMP_METRICS_DIR, "fgsm_eval_resource_trace.csv")
FGSM_EVAL_SUMMARY_CSV      = os.path.join(COMP_METRICS_DIR, "fgsm_eval_resource_summary.csv")
FGSM_XFER_TRACE_CSV        = os.path.join(COMP_METRICS_DIR, "fgsm_transfer_resource_trace.csv")
FGSM_XFER_SUMMARY_CSV      = os.path.join(COMP_METRICS_DIR, "fgsm_transfer_resource_summary.csv")

# bim result files
BIM_SINGLE_RUN_CSV         = os.path.join(BIM_RESULTS_DIR, "bim_morphence_single_run.csv")
BIM_STATS_CSV              = os.path.join(BIM_RESULTS_DIR, "bim_morphence_single_pass_stats.csv")
BIM_WEIGHT_DIVERSITY_CSV   = os.path.join(BIM_RESULTS_DIR, "bim_weight_diversity.csv")
BIM_PRED_DIVERSITY_CSV     = os.path.join(BIM_RESULTS_DIR, "bim_prediction_diversity.csv")
BIM_XFER_MATRIX_CSV        = os.path.join(BIM_RESULTS_DIR, "bim_transfer_matrix.csv")
BIM_EVAL_TRACE_CSV         = os.path.join(COMP_METRICS_DIR, "bim_eval_resource_trace.csv")
BIM_EVAL_SUMMARY_CSV       = os.path.join(COMP_METRICS_DIR, "bim_eval_resource_summary.csv")
BIM_XFER_TRACE_CSV         = os.path.join(COMP_METRICS_DIR, "bim_transfer_resource_trace.csv")
BIM_XFER_SUMMARY_CSV       = os.path.join(COMP_METRICS_DIR, "bim_transfer_resource_summary.csv")

# pgd result files
PGD_SINGLE_RUN_CSV         = os.path.join(PGD_RESULTS_DIR, "pgd_morphence_single_run.csv")
PGD_STATS_CSV              = os.path.join(PGD_RESULTS_DIR, "pgd_morphence_single_pass_stats.csv")
PGD_WEIGHT_DIVERSITY_CSV   = os.path.join(PGD_RESULTS_DIR, "pgd_weight_diversity.csv")
PGD_PRED_DIVERSITY_CSV     = os.path.join(PGD_RESULTS_DIR, "pgd_prediction_diversity.csv")
PGD_XFER_MATRIX_CSV        = os.path.join(PGD_RESULTS_DIR, "pgd_transfer_matrix.csv")
PGD_EVAL_TRACE_CSV         = os.path.join(COMP_METRICS_DIR, "pgd_eval_resource_trace.csv")
PGD_EVAL_SUMMARY_CSV       = os.path.join(COMP_METRICS_DIR, "pgd_eval_resource_summary.csv")
PGD_XFER_TRACE_CSV         = os.path.join(COMP_METRICS_DIR, "pgd_transfer_resource_trace.csv")
PGD_XFER_SUMMARY_CSV       = os.path.join(COMP_METRICS_DIR, "pgd_transfer_resource_summary.csv")

print("PROJECT_DIR   :", PROJECT_DIR)
print("BASE_MODEL_DIR:", BASE_MODEL_DIR)
print("DATASET_PATH  :", DATASET_PATH)

PROJECT_DIR   : /content/drive/MyDrive/298/Elec/proj_v3
BASE_MODEL_DIR: /content/drive/MyDrive/298/Elec/proj_v3/baseModel
DATASET_PATH  : /content/drive/MyDrive/298/Elec/Data/LD2011_2014.txt


In [ ]:
# =========================================================
# CONFIG
# =========================================================

FREQ = "15min"
STEPS_PER_HOUR = 4
HOURS_PER_DAY  = 24

HIST_HOURS  = 96
HZN_HOURS   = 24
HIST_STEPS  = HIST_HOURS * STEPS_PER_HOUR   # 384
HZN_STEPS   = HZN_HOURS  * STEPS_PER_HOUR   # 96
SLIDE       = 1

TRAIN_FRAC  = 0.70
VAL_FRAC    = 0.10

BATCH       = 16
PATCH       = 4
D_FEATS     = 1
D_MODEL     = 256
N_HEAD      = 4
N_LAYERS    = 3
DIM_FF      = 512
DROPOUT     = 0.1

LR_BASE     = 5e-4
LR_STUDENT  = 1e-4
EPOCHS_BASE = 30
EPOCHS_STU  = 5

N_STUDENTS = 4

FGSM_TRAIN_EPS_LIST = [0.1, 0.2, 0.3, 0.5]
FGSM_EVAL_EPS_LIST  = [0.1, 0.2, 0.3, 0.5]
RFGSM_ALPHA_FACTOR  = 1.0

BIM_TRAIN_EPS_LIST = [0.1, 0.2, 0.3, 0.5]
BIM_EVAL_EPS_LIST  = [0.1, 0.2, 0.3, 0.5]
BIM_ITERS          = 10
BIM_RANDOM_START_TRAIN = False
BIM_RANDOM_START_EVAL  = True

PGD_TRAIN_EPS_LIST = [0.1, 0.2, 0.3, 0.5]
PGD_EVAL_EPS_LIST  = [0.1, 0.2, 0.3, 0.5]
PGD_ITERS          = 10
PGD_RANDOM_START_TRAIN = True
PGD_RANDOM_START_EVAL  = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

Device: cuda


In [ ]:
# =========================================================
# HEARTBEAT
# =========================================================

HEARTBEAT_SECS = 600

def start_heartbeat(name="task"):
    flag = {"on": True}
    start_time = time.time()

    def _hb():
        while flag["on"]:
            time.sleep(HEARTBEAT_SECS)
            if flag["on"]:
                elapsed = (time.time() - start_time) / 60.0
                print(f"[heartbeat:{name}] running... elapsed={elapsed:.2f} min", flush=True)

    t = threading.Thread(target=_hb, daemon=True)
    t.start()
    return flag, t

def stop_heartbeat(flag, thread):
    flag["on"] = False
    try:
        thread.join(timeout=1)
    except Exception:
        pass

In [ ]:
# =========================================================
# COMPUTATIONAL METRICS HELPERS
# =========================================================

class ResourceMonitor:
    def __init__(self, sample_interval=2.0):
        self.sample_interval = sample_interval
        self._stop_event = threading.Event()
        self._thread = None
        self.records = []
        self.max_cpu_ram_mb = 0.0
        self.max_gpu_mem_mb = 0.0

    def _query_gpu(self):
        try:
            cmd = [
                "nvidia-smi",
                "--query-gpu=utilization.gpu,memory.used,memory.total,power.draw",
                "--format=csv,noheader,nounits"
            ]
            out = subprocess.check_output(cmd).decode("utf-8").strip()
            vals = [v.strip() for v in out.split(",")]

            gpu_util = float(vals[0]) if vals[0] not in ["N/A", "[Not Supported]"] else None
            mem_used = float(vals[1]) if vals[1] not in ["N/A", "[Not Supported]"] else None
            mem_total = float(vals[2]) if vals[2] not in ["N/A", "[Not Supported]"] else None
            power_w = float(vals[3]) if vals[3] not in ["N/A", "[Not Supported]"] else None

            return {
                "gpu_util_percent": gpu_util,
                "gpu_mem_used_mb": mem_used,
                "gpu_mem_total_mb": mem_total,
                "gpu_power_w": power_w,
            }
        except Exception:
            return {
                "gpu_util_percent": None,
                "gpu_mem_used_mb": None,
                "gpu_mem_total_mb": None,
                "gpu_power_w": None,
            }

    def _sample_once(self):
        ts = time.time()

        proc = psutil.Process(os.getpid())
        cpu_ram_mb = proc.memory_info().rss / (1024 ** 2)
        self.max_cpu_ram_mb = max(self.max_cpu_ram_mb, cpu_ram_mb)

        torch_gpu_mem_mb = None
        if torch.cuda.is_available():
            try:
                torch_gpu_mem_mb = torch.cuda.memory_allocated() / (1024 ** 2)
                self.max_gpu_mem_mb = max(self.max_gpu_mem_mb, torch_gpu_mem_mb)
            except Exception:
                pass

        gpu_stats = self._query_gpu()
        if gpu_stats["gpu_mem_used_mb"] is not None:
            self.max_gpu_mem_mb = max(self.max_gpu_mem_mb, gpu_stats["gpu_mem_used_mb"])

        self.records.append({
            "timestamp": ts,
            "cpu_ram_mb": cpu_ram_mb,
            "torch_gpu_mem_mb": torch_gpu_mem_mb,
            **gpu_stats
        })

    def _run(self):
        while not self._stop_event.is_set():
            self._sample_once()
            time.sleep(self.sample_interval)

    def start(self):
        self.records = []
        self.max_cpu_ram_mb = 0.0
        self.max_gpu_mem_mb = 0.0
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join(timeout=2)

    def to_dataframe(self):
        return pd.DataFrame(self.records)

    def summary(self):
        if len(self.records) == 0:
            return {}

        dfm = pd.DataFrame(self.records).sort_values("timestamp").reset_index(drop=True)

        avg_gpu_util = None
        gpu_active_pct = None
        if dfm["gpu_util_percent"].notna().any():
            avg_gpu_util = float(dfm["gpu_util_percent"].dropna().mean())
            gpu_active_pct = float((dfm["gpu_util_percent"] > 0).mean() * 100.0)

        est_gpu_energy_j = None
        if dfm["gpu_power_w"].notna().sum() >= 2:
            t = dfm["timestamp"].values
            p = dfm["gpu_power_w"].astype(float).values
            mask = ~np.isnan(p)
            if mask.sum() >= 2:
                est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))

        return {
            "n_samples": int(len(dfm)),
            "sample_interval_sec": self.sample_interval,
            "max_cpu_ram_mb": float(self.max_cpu_ram_mb),
            "max_gpu_mem_mb": float(self.max_gpu_mem_mb),
            "avg_gpu_util_percent": avg_gpu_util,
            "gpu_active_percent_of_samples": gpu_active_pct,
            "est_gpu_energy_j": est_gpu_energy_j,
        }

def save_monitor_outputs(monitor, trace_csv, summary_csv, extra_summary=None):
    trace_df = monitor.to_dataframe()
    summary = monitor.summary()

    if extra_summary is not None:
        summary.update(extra_summary)

    trace_df.to_csv(trace_csv, index=False)
    pd.DataFrame([summary]).to_csv(summary_csv, index=False)

    print("Saved resource trace to:", trace_csv)
    print("Saved resource summary to:", summary_csv)
    print("Resource summary:", summary)

    return trace_df, summary

def append_summary_row(row, out_csv):
    if os.path.exists(out_csv):
        old = pd.read_csv(out_csv)
        old = pd.concat([old, pd.DataFrame([row])], ignore_index=True)
        old.to_csv(out_csv, index=False)
    else:
        pd.DataFrame([row]).to_csv(out_csv, index=False)

def profile_model_once(model, sample_input, out_csv, model_label="base"):
    model.eval()
    with torch.no_grad():
        macs, params = profile(model, inputs=(sample_input,), verbose=False)

    flops_est = 2 * macs

    row = {
        "model": model_label,
        "estimated_macs": float(macs),
        "estimated_flops": float(flops_est),
        "parameter_count": float(params),
    }

    pd.DataFrame([row]).to_csv(out_csv, index=False)
    print("Saved model profile to:", out_csv)
    print(row)
    return row

In [ ]:
# =========================================================
# DATA LOADING + PREPROCESSING
# =========================================================

print("Reading raw txt:", DATASET_PATH)

df = pd.read_csv(
    DATASET_PATH,
    sep=';',
    quotechar='"',
    decimal=',',
    parse_dates=[0],
    index_col=0,
    na_values=['', 'NA', 'NaN'],
    low_memory=False
)
df.index.name = "timestamp"
df = df.sort_index()
print("Raw shape:", df.shape)

METER = "MT_001"
START_DT = pd.Timestamp("2013-01-01 00:00:00")
END_DT   = pd.Timestamp("2013-12-31 23:45:00")

df = df[[METER]]
df = df.loc[(df.index >= START_DT) & (df.index <= END_DT)].copy()

full_idx = pd.date_range(START_DT, END_DT, freq=FREQ)
df = df.reindex(full_idx)

df = df.apply(pd.to_numeric, errors='coerce')
df = df.ffill(limit=4).bfill(limit=4)
print("Post filtering shape:", df.shape)

n_t       = len(df)
split_idx = int(TRAIN_FRAC * n_t)
df_train  = df.iloc[:split_idx]
df_all    = df

scaler = MinMaxScaler()
train_scaled = pd.DataFrame(
    scaler.fit_transform(df_train),
    index=df_train.index,
    columns=df_train.columns
)
full_scaled = pd.DataFrame(
    scaler.transform(df_all),
    index=df_all.index,
    columns=df_all.columns
)
print("Scaled shape:", full_scaled.shape)

values = full_scaled.values.astype(np.float32)
T, D = values.shape
assert D == 1, f"Expected 1 feature, got {D}"
print("T, D:", T, D)

X_list, Y_list, meta = [], [], []
for t in range(HIST_STEPS, T - HZN_STEPS, SLIDE):
    x = values[t - HIST_STEPS:t, :]
    y = values[t:t + HZN_STEPS, :]
    X_list.append(x)
    Y_list.append(y)
    meta.append((
        full_scaled.index[t - HIST_STEPS],
        full_scaled.index[t - 1],
        full_scaled.index[t],
        full_scaled.index[t + HZN_STEPS - 1],
    ))

X = np.stack(X_list)
Y = np.stack(Y_list)
meta_df = pd.DataFrame(meta, columns=["x_start", "x_end", "y_start", "y_end"])

print("Windowed tensors:", X.shape, Y.shape)

N = len(X)
n_train = int(TRAIN_FRAC * N)
n_val   = int(VAL_FRAC * N)
n_test  = N - n_train - n_val

idx_train = slice(0, n_train)
idx_val   = slice(n_train, n_train + n_val)
idx_test  = slice(n_train + n_val, N)

X_train, Y_train = X[idx_train], Y[idx_train]
X_val,   Y_val   = X[idx_val],   Y[idx_val]
X_test,  Y_test  = X[idx_test],  Y[idx_test]

meta_train = meta_df.iloc[idx_train].reset_index(drop=True)
meta_val   = meta_df.iloc[idx_val].reset_index(drop=True)
meta_test  = meta_df.iloc[idx_test].reset_index(drop=True)

print("Splits:")
print("  train:", X_train.shape)
print("  val  :", X_val.shape)
print("  test :", X_test.shape)

np.save(os.path.join(COMMON_ARTIFACTS_DIR, "X_train.npy"), X_train)
np.save(os.path.join(COMMON_ARTIFACTS_DIR, "Y_train.npy"), Y_train)
np.save(os.path.join(COMMON_ARTIFACTS_DIR, "X_val.npy"), X_val)
np.save(os.path.join(COMMON_ARTIFACTS_DIR, "Y_val.npy"), Y_val)
np.save(os.path.join(COMMON_ARTIFACTS_DIR, "X_test.npy"), X_test)
np.save(os.path.join(COMMON_ARTIFACTS_DIR, "Y_test.npy"), Y_test)

meta_train.to_csv(os.path.join(COMMON_ARTIFACTS_DIR, "meta_train.csv"), index=False)
meta_val.to_csv(os.path.join(COMMON_ARTIFACTS_DIR, "meta_val.csv"), index=False)
meta_test.to_csv(os.path.join(COMMON_ARTIFACTS_DIR, "meta_test.csv"), index=False)

print("Saved preprocessing artifacts to:", COMMON_ARTIFACTS_DIR)

Reading raw txt: /content/drive/MyDrive/298/Elec/Data/LD2011_2014.txt
Raw shape: (140256, 370)
Post filtering shape: (35040, 1)
Scaled shape: (35040, 1)
T, D: 35040 1
Windowed tensors: (34560, 384, 1) (34560, 96, 1)
Splits:
  train: (24192, 384, 1)
  val  : (3456, 384, 1)
  test : (6912, 384, 1)
Saved preprocessing artifacts to: /content/drive/MyDrive/298/Elec/proj_v3/common_artifacts


In [ ]:
# =========================================================
# DATALOADERS
# =========================================================

class ElecSeq2SeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)
        self.Y = torch.from_numpy(Y)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.Y[i]

train_dl = DataLoader(
    ElecSeq2SeqDataset(X_train, Y_train),
    batch_size=BATCH,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_dl = DataLoader(
    ElecSeq2SeqDataset(X_val, Y_val),
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_dl = DataLoader(
    ElecSeq2SeqDataset(X_test, Y_test),
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

xb, yb = next(iter(train_dl))
print("One batch X:", xb.shape)
print("One batch Y:", yb.shape)

One batch X: torch.Size([16, 384, 1])
One batch Y: torch.Size([16, 96, 1])


In [ ]:
# =========================================================
# PATCHING + MODEL
# =========================================================

def patchify(x, patch):
    if patch == 1:
        return x
    B, T, D = x.shape
    T_trim = (T // patch) * patch
    x = x[:, :T_trim, :].reshape(B, T_trim // patch, patch, D).mean(dim=2)
    return x

def unpatchify(y_patch, patch, T_out):
    if patch == 1:
        return y_patch[:, :T_out, :]
    y = y_patch.repeat_interleave(patch, dim=1)
    return y[:, :T_out, :]

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=6000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class ElecSeq2SeqTransformer(nn.Module):
    def __init__(self,
                 d_feats,
                 d_model=256,
                 nhead=4,
                 num_layers=3,
                 dim_ff=512,
                 dropout=0.1,
                 hist_steps=HIST_STEPS,
                 hzn_steps=HZN_STEPS,
                 patch=PATCH):
        super().__init__()
        self.d_feats    = d_feats
        self.d_model    = d_model
        self.hist_steps = hist_steps
        self.hzn_steps  = hzn_steps
        self.patch      = patch

        self.in_proj = nn.Linear(d_feats, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.enc_pos = PositionalEncoding(d_model, max_len=(hist_steps // patch) + 16)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_layers)
        self.dec_pos = PositionalEncoding(d_model, max_len=(hzn_steps // patch) + 16)

        self.out_proj = nn.Linear(d_model, d_feats)

        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        B, T, D = x.shape
        x_p = patchify(x, self.patch)
        z   = self.in_proj(x_p)
        z   = self.enc_pos(z)
        mem = self.encoder(z)

        dec_Tp = self.hzn_steps // self.patch
        tgt = torch.zeros(B, dec_Tp, self.d_model, device=x.device)
        tgt = self.dec_pos(tgt)
        out = self.decoder(tgt, mem)
        y_p = self.out_proj(out)
        y   = unpatchify(y_p, self.patch, self.hzn_steps)
        return y

model = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to(device)

with torch.no_grad():
    pred = model(xb.to(device))

print("Pred batch shape:", pred.shape)

Pred batch shape: torch.Size([16, 96, 1])


/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


In [ ]:
# =========================================================
# LOSSES + METRICS
# =========================================================

mse_loss = nn.MSELoss()

def rmse_only(pred, true):
    mse = mse_loss(pred, true)
    return torch.sqrt(mse + 1e-12)

def seq_rmse_np(y_true_np, y_pred_np):
    return float(np.sqrt(mean_squared_error(y_true_np.ravel(), y_pred_np.ravel())))

In [ ]:
# =========================================================
# ATTACKS
# =========================================================

def rfgsm_attack(attacker_model, X, Y, eps, alpha=None):
    attacker_model.eval()
    if alpha is None:
        alpha = eps * RFGSM_ALPHA_FACTOR

    X0 = (X + torch.empty_like(X).uniform_(-eps, eps)).clamp(0.0, 1.0)
    X0 = X0.detach().requires_grad_(True)

    pred = attacker_model(X0)
    loss = mse_loss(pred, Y)
    grad = torch.autograd.grad(loss, X0, only_inputs=True, retain_graph=False)[0]

    with torch.no_grad():
        X_adv = (X0 + alpha * grad.sign()).clamp(0.0, 1.0)

    return X_adv.detach()

def bim_attack(attacker_model, X, Y, eps, alpha, iters, random_start=False):
    attacker_model.eval()

    if random_start:
        noise = torch.empty_like(X).uniform_(-eps, eps)
        X_adv = (X + noise).clamp(0.0, 1.0)
    else:
        X_adv = X.clone()

    X_adv = X_adv.detach().requires_grad_(True)

    for _ in range(iters):
        preds = attacker_model(X_adv)
        loss = mse_loss(preds, Y)

        if X_adv.grad is not None:
            X_adv.grad.zero_()

        loss.backward()
        grad = X_adv.grad

        X_adv = (X_adv + alpha * grad.sign()).detach()
        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0.0, 1.0).detach().requires_grad_(True)

    return X_adv.detach()

def pgd_attack(attacker_model, X, Y, eps, alpha, iters, random_start=True):
    attacker_model.eval()

    if random_start:
        noise = torch.empty_like(X).uniform_(-eps, eps)
        X_adv = (X + noise).clamp(0.0, 1.0)
    else:
        X_adv = X.clone()

    X_adv = X_adv.detach().requires_grad_(True)

    for _ in range(iters):
        preds = attacker_model(X_adv)
        loss = mse_loss(preds, Y)

        if X_adv.grad is not None:
            X_adv.grad.zero_()

        loss.backward()
        grad = X_adv.grad

        X_adv = (X_adv + alpha * grad.sign()).detach()
        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0.0, 1.0).detach().requires_grad_(True)

    return X_adv.detach()

In [ ]:
# =========================================================
# TRAIN / EVAL HELPERS
# =========================================================

def train_base(model, train_loader, val_loader, epochs=EPOCHS_BASE, lr=LR_BASE):
    opt = optim.AdamW(model.parameters(), lr=lr)
    best = float("inf")
    log_rows = []

    hb_flag, hb_thread = start_heartbeat("base_train")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_time = time.time()

    monitor.start()

    try:
        for ep in range(1, epochs + 1):
            model.train()
            tot = cnt = 0
            ep_start = time.time()

            for xb, yb in train_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                pred = model(xb)
                loss = rmse_only(pred, yb)

                opt.zero_grad()
                loss.backward()
                opt.step()

                bs = xb.size(0)
                tot += loss.item() * bs
                cnt += bs

            train_loss = tot / max(1, cnt)

            model.eval()
            vt = vc = 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb = xb.to(device, non_blocking=True)
                    yb = yb.to(device, non_blocking=True)
                    pred = model(xb)
                    loss = rmse_only(pred, yb)
                    vt += loss.item() * xb.size(0)
                    vc += xb.size(0)

            val_loss = vt / max(1, vc)

            ep_mins = (time.time() - ep_start) / 60.0
            elapsed = (time.time() - start_time) / 60.0

            print(
                f"[Base] Epoch {ep:02d} | train {train_loss:.4f} | "
                f"val {val_loss:.4f} | epoch {ep_mins:.2f} min | elapsed {elapsed:.2f} min"
            )

            log_rows.append({
                "epoch": ep,
                "train_rmse": train_loss,
                "val_rmse": val_loss,
                "epoch_time_min": ep_mins,
                "elapsed_time_min": elapsed,
            })
            pd.DataFrame(log_rows).to_csv(BASE_TRAIN_LOG_CSV, index=False)

            if val_loss < best:
                best = val_loss
                torch.save(model.state_dict(), BASE_CKPT_PATH)
                print("  [Base] Saved best to", BASE_CKPT_PATH, flush=True)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    _, resource_summary = save_monitor_outputs(
        monitor,
        trace_csv=BASE_RESOURCE_TRACE_CSV,
        summary_csv=BASE_RESOURCE_SUMMARY_CSV,
        extra_summary={
            "stage": "base_training",
            "epochs": epochs,
            "learning_rate": lr,
            "total_elapsed_minutes": (time.time() - start_time) / 60.0,
        }
    )

    return BASE_CKPT_PATH, pd.DataFrame(log_rows), resource_summary

@torch.no_grad()
def eval_one_epoch(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        p = model(xb)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

@torch.no_grad()
def eval_clean_rmse(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        p = model(xb)
        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())
    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

@torch.no_grad()
def eval_ensemble_rmse(models, loader):
    for m in models:
        m.eval()

    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        preds = []
        for m in models:
            preds.append(m(xb))
        preds = torch.stack(preds, dim=0).mean(dim=0)

        ys.append(yb.cpu().numpy())
        ps.append(preds.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

def eval_pair_attack(defender, attacker, loader, attack_fn, atk_kwargs):
    defender.eval()
    attacker.eval()

    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        xa = attack_fn(attacker, xb, yb, **atk_kwargs)

        with torch.no_grad():
            p = defender(xa)

        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

def eval_random_switch_attack(models, loader, attack_fn, atk_kwargs_func):
    for m in models:
        m.eval()

    ys, ps = [], []
    n_stu = len(models)

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        atk_idx = random.randrange(n_stu)
        def_idx = random.randrange(n_stu)

        attacker = models[atk_idx]
        defender = models[def_idx]

        atk_kwargs = atk_kwargs_func()
        xa = attack_fn(attacker, xb, yb, **atk_kwargs)

        with torch.no_grad():
            p = defender(xa)

        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return seq_rmse_np(ys, ps)

In [ ]:
# =========================================================
# BASE TRAINING
# =========================================================

print("\nTraining base model")

base = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to(device)

base_ckpt, base_train_log_df, base_resource_summary = train_base(base, train_dl, val_dl)
print("Base checkpoint:", base_ckpt)

best_base = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to(device)
best_base.load_state_dict(torch.load(base_ckpt, map_location=device))
best_base.eval()

val_rmse_best = eval_one_epoch(best_base, val_dl)
test_rmse_best = eval_one_epoch(best_base, test_dl)

print(f"Best model val RMSE : {val_rmse_best:.5f}")
print(f"Best model test RMSE: {test_rmse_best:.5f}")

dummy_x = torch.randn(1, HIST_STEPS, D_FEATS).to(device)
base_profile = profile_model_once(
    model=best_base,
    sample_input=dummy_x,
    out_csv=BASE_PROFILE_CSV,
    model_label="elec_base_transformer"
)


Training base model


/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[Base] Epoch 01 | train 0.3376 | val 0.1806 | epoch 0.17 min | elapsed 0.17 min
  [Base] Saved best to /content/drive/MyDrive/298/Elec/proj_v3/baseModel/elec_base_best.pth
[Base] Epoch 02 | train 0.1312 | val 0.1182 | epoch 0.17 min | elapsed 0.34 min
  [Base] Saved best to /content/drive/MyDrive/298/Elec/proj_v3/baseModel/elec_base_best.pth
[Base] Epoch 03 | train 0.1173 | val 0.1095 | epoch 0.16 min | elapsed 0.50 min
  [Base] Saved best to /content/drive/MyDrive/298/Elec/proj_v3/baseModel/elec_base_best.pth
[Base] Epoch 04 | train 0.1115 | val 0.1075 | epoch 0.17 min | elapsed 0.67 min
  [Base] Saved best to /content/drive/MyDrive/298/Elec/proj_v3/baseModel/elec_base_best.pth
[Base] Epoch 05 | train 0.1069 | val 0.1128 | epoch 0.16 min | elapsed 0.84 min
[Base] Epoch 06 | train 0.1026 | val 0.1213 | epoch 0.16 min | elapsed 1.00 min
[Base] Epoch 07 | train 0.0983 | val 0.1165 | epoch 0.17 min | elapsed 1.16 min
[Base] Epoch 08 | train 0.0952 | val 0.1377 | epoch 0.17 min | elapsed 1

/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))
/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Best model val RMSE : 0.11522
Best model test RMSE: 0.11526
Saved model profile to: /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/base_model_profile.csv
{'model': 'elec_base_transformer', 'estimated_macs': 95213568.0, 'estimated_flops': 190427136.0, 'parameter_count': 1585921.0}


In [ ]:
# =========================================================
# STUDENT GENERATION HELPERS
# =========================================================

def generate_students_from_base(base_ckpt_path, out_dir, n_students=N_STUDENTS, noise_std=1e-3):
    os.makedirs(out_dir, exist_ok=True)

    base_cpu = ElecSeq2SeqTransformer(
        d_feats=D_FEATS,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_layers=N_LAYERS,
        dim_ff=DIM_FF,
        dropout=DROPOUT,
        hist_steps=HIST_STEPS,
        hzn_steps=HZN_STEPS,
        patch=PATCH
    ).to("cpu")
    base_cpu.load_state_dict(torch.load(base_ckpt_path, map_location="cpu"))

    student_paths = []

    print(f"\nCreating students in: {out_dir}")
    for i in range(1, n_students + 1):
        st = copy.deepcopy(base_cpu)
        with torch.no_grad():
            for _, p in st.named_parameters():
                p.add_(noise_std * torch.randn_like(p))

        outp = os.path.join(out_dir, f"student_{i:02d}.pth")
        torch.save(st.state_dict(), outp)
        student_paths.append(outp)
        print("Saved student:", outp)

    return sorted(student_paths)

def load_student_models(path_list):
    models = []
    names = []

    for sp in sorted(path_list):
        stu = ElecSeq2SeqTransformer(
            d_feats=D_FEATS,
            d_model=D_MODEL,
            nhead=N_HEAD,
            num_layers=N_LAYERS,
            dim_ff=DIM_FF,
            dropout=DROPOUT,
            hist_steps=HIST_STEPS,
            hzn_steps=HZN_STEPS,
            patch=PATCH
        ).to(device)

        stu.load_state_dict(torch.load(sp, map_location=device))
        stu.eval()

        models.append(stu)
        names.append(os.path.basename(sp).replace(".pth", ""))

    return models, names

In [ ]:
# =========================================================
# GENERATE SEPARATE BASE STUDENT POOLS FOR FGSM / BIM / PGD
# =========================================================

fgsm_student_paths = generate_students_from_base(BASE_CKPT_PATH, FGSM_STUDENTS_DIR, n_students=N_STUDENTS)
bim_student_paths  = generate_students_from_base(BASE_CKPT_PATH, BIM_STUDENTS_DIR,  n_students=N_STUDENTS)
pgd_student_paths  = generate_students_from_base(BASE_CKPT_PATH, PGD_STUDENTS_DIR,  n_students=N_STUDENTS)

/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)



Creating students in: /content/drive/MyDrive/298/Elec/proj_v3/fgsm/students
Saved student: /content/drive/MyDrive/298/Elec/proj_v3/fgsm/students/student_01.pth
Saved student: /content/drive/MyDrive/298/Elec/proj_v3/fgsm/students/student_02.pth
Saved student: /content/drive/MyDrive/298/Elec/proj_v3/fgsm/students/student_03.pth
Saved student: /content/drive/MyDrive/298/Elec/proj_v3/fgsm/students/student_04.pth

Creating students in: /content/drive/MyDrive/298/Elec/proj_v3/bim/students
Saved student: /content/drive/MyDrive/298/Elec/proj_v3/bim/students/student_01.pth
Saved student: /content/drive/MyDrive/298/Elec/proj_v3/bim/students/student_02.pth
Saved student: /content/drive/MyDrive/298/Elec/proj_v3/bim/students/student_03.pth
Saved student: /content/drive/MyDrive/298/Elec/proj_v3/bim/students/student_04.pth

Creating students in: /content/drive/MyDrive/298/Elec/proj_v3/pgd/students
Saved student: /content/drive/MyDrive/298/Elec/proj_v3/pgd/students/student_01.pth
Saved student: /cont

In [ ]:
# =========================================================
# ADV TRAINING HELPERS FOR ALL 3 ATTACKS
# =========================================================

def adv_train_students_fgsm(student_paths):
    adv_paths = []

    print("\nTraining FGSM adv students")
    for sp in student_paths:
        idx = os.path.basename(sp).split("_")[1].split(".")[0]
        print(f"\n[FGSM Student {idx}] adversarial training...")

        stu = ElecSeq2SeqTransformer(
            d_feats=D_FEATS,
            d_model=D_MODEL,
            nhead=N_HEAD,
            num_layers=N_LAYERS,
            dim_ff=DIM_FF,
            dropout=DROPOUT,
            hist_steps=HIST_STEPS,
            hzn_steps=HZN_STEPS,
            patch=PATCH
        ).to(device)
        stu.load_state_dict(torch.load(sp, map_location=device))

        opt = optim.AdamW(stu.parameters(), lr=LR_STUDENT)

        hb_flag, hb_thread = start_heartbeat(f"student_{idx}_fgsm_train")
        monitor = ResourceMonitor(sample_interval=2.0)
        start_time = time.time()
        history = []

        monitor.start()

        try:
            for ep in range(1, EPOCHS_STU + 1):
                stu.train()
                tot = cnt = 0
                ep_start = time.time()

                for xb, yb in train_dl:
                    xb = xb.to(device, non_blocking=True)
                    yb = yb.to(device, non_blocking=True)

                    pred_c = stu(xb)
                    Lc = rmse_only(pred_c, yb)

                    adv_losses = []
                    for eps in FGSM_TRAIN_EPS_LIST:
                        xa = rfgsm_attack(stu, xb, yb, eps=eps)
                        pred_a = stu(xa)
                        La = rmse_only(pred_a, yb)
                        adv_losses.append(La)

                    La_mean = torch.stack(adv_losses).mean() if len(adv_losses) > 0 else 0.0 * Lc
                    loss = 0.25 * Lc + 0.75 * La_mean

                    opt.zero_grad()
                    loss.backward()
                    opt.step()

                    bs = xb.size(0)
                    tot += loss.item() * bs
                    cnt += bs

                avg_loss = tot / max(1, cnt)
                ep_mins  = (time.time() - ep_start) / 60.0
                elapsed  = (time.time() - start_time) / 60.0

                history.append({
                    "student_idx": idx,
                    "epoch": ep,
                    "avg_loss": avg_loss,
                    "epoch_time_min": ep_mins,
                    "elapsed_time_min": elapsed,
                })

                print(
                    f"  [FGSM Student {idx}] Epoch {ep:02d} | avg_loss={avg_loss:.4f} "
                    f"| epoch={ep_mins:.2f} min | elapsed={elapsed:.2f} min"
                )

        finally:
            monitor.stop()
            stop_heartbeat(hb_flag, hb_thread)

        hist_path = os.path.join(FGSM_LOGS_DIR, f"student_{idx}_fgsm_train_log.csv")
        pd.DataFrame(history).to_csv(hist_path, index=False)

        outp = os.path.join(FGSM_STUDENTS_ADV_DIR, f"student_{idx}_fgsm_adv.pth")
        torch.save(stu.to("cpu").state_dict(), outp)
        adv_paths.append(outp)
        print(f"  [FGSM Student {idx}] Saved adv student:", outp)

        summary_row = monitor.summary()
        summary_row.update({
            "student": f"student_{idx}",
            "attack_train_type": "fgsm",
            "epochs": EPOCHS_STU,
            "learning_rate": LR_STUDENT,
            "total_elapsed_minutes": (time.time() - start_time) / 60.0,
        })
        append_summary_row(summary_row, FGSM_TRAIN_RESOURCE_CSV)

    return sorted(adv_paths)

def adv_train_students_bim(student_paths):
    adv_paths = []

    print("\nTraining BIM adv students")
    for sp in student_paths:
        idx = os.path.basename(sp).split("_")[1].split(".")[0]
        print(f"\n[BIM Student {idx}] adversarial training...")

        stu = ElecSeq2SeqTransformer(
            d_feats=D_FEATS,
            d_model=D_MODEL,
            nhead=N_HEAD,
            num_layers=N_LAYERS,
            dim_ff=DIM_FF,
            dropout=DROPOUT,
            hist_steps=HIST_STEPS,
            hzn_steps=HZN_STEPS,
            patch=PATCH
        ).to(device)
        stu.load_state_dict(torch.load(sp, map_location=device))

        opt = optim.AdamW(stu.parameters(), lr=LR_STUDENT)

        hb_flag, hb_thread = start_heartbeat(f"student_{idx}_bim_train")
        monitor = ResourceMonitor(sample_interval=2.0)
        start_time = time.time()
        history = []

        monitor.start()

        try:
            for ep in range(1, EPOCHS_STU + 1):
                stu.train()
                tot = cnt = 0
                ep_start = time.time()

                for xb, yb in train_dl:
                    xb = xb.to(device, non_blocking=True)
                    yb = yb.to(device, non_blocking=True)

                    pred_c = stu(xb)
                    Lc = rmse_only(pred_c, yb)

                    adv_losses = []
                    for eps in BIM_TRAIN_EPS_LIST:
                        xa = bim_attack(
                            stu, xb, yb,
                            eps=eps,
                            alpha=eps / BIM_ITERS,
                            iters=BIM_ITERS,
                            random_start=BIM_RANDOM_START_TRAIN
                        )
                        pred_a = stu(xa)
                        La = rmse_only(pred_a, yb)
                        adv_losses.append(La)

                    La_mean = torch.stack(adv_losses).mean() if len(adv_losses) > 0 else 0.0 * Lc
                    loss = 0.25 * Lc + 0.75 * La_mean

                    opt.zero_grad()
                    loss.backward()
                    opt.step()

                    bs = xb.size(0)
                    tot += loss.item() * bs
                    cnt += bs

                avg_loss = tot / max(1, cnt)
                ep_mins  = (time.time() - ep_start) / 60.0
                elapsed  = (time.time() - start_time) / 60.0

                history.append({
                    "student_idx": idx,
                    "epoch": ep,
                    "avg_loss": avg_loss,
                    "epoch_time_min": ep_mins,
                    "elapsed_time_min": elapsed,
                })

                print(
                    f"  [BIM Student {idx}] Epoch {ep:02d} | avg_loss={avg_loss:.4f} "
                    f"| epoch={ep_mins:.2f} min | elapsed={elapsed:.2f} min"
                )

        finally:
            monitor.stop()
            stop_heartbeat(hb_flag, hb_thread)

        hist_path = os.path.join(BIM_LOGS_DIR, f"student_{idx}_bim_train_log.csv")
        pd.DataFrame(history).to_csv(hist_path, index=False)

        outp = os.path.join(BIM_STUDENTS_ADV_DIR, f"student_{idx}_bim_adv.pth")
        torch.save(stu.to("cpu").state_dict(), outp)
        adv_paths.append(outp)
        print(f"  [BIM Student {idx}] Saved adv student:", outp)

        summary_row = monitor.summary()
        summary_row.update({
            "student": f"student_{idx}",
            "attack_train_type": "bim",
            "epochs": EPOCHS_STU,
            "learning_rate": LR_STUDENT,
            "iters": BIM_ITERS,
            "total_elapsed_minutes": (time.time() - start_time) / 60.0,
        })
        append_summary_row(summary_row, BIM_TRAIN_RESOURCE_CSV)

    return sorted(adv_paths)

def adv_train_students_pgd(student_paths):
    adv_paths = []

    print("\nTraining PGD adv students")
    for sp in student_paths:
        idx = os.path.basename(sp).split("_")[1].split(".")[0]
        print(f"\n[PGD Student {idx}] adversarial training...")

        stu = ElecSeq2SeqTransformer(
            d_feats=D_FEATS,
            d_model=D_MODEL,
            nhead=N_HEAD,
            num_layers=N_LAYERS,
            dim_ff=DIM_FF,
            dropout=DROPOUT,
            hist_steps=HIST_STEPS,
            hzn_steps=HZN_STEPS,
            patch=PATCH
        ).to(device)
        stu.load_state_dict(torch.load(sp, map_location=device))

        opt = optim.AdamW(stu.parameters(), lr=LR_STUDENT)

        hb_flag, hb_thread = start_heartbeat(f"student_{idx}_pgd_train")
        monitor = ResourceMonitor(sample_interval=2.0)
        start_time = time.time()
        history = []

        monitor.start()

        try:
            for ep in range(1, EPOCHS_STU + 1):
                stu.train()
                tot = cnt = 0
                ep_start = time.time()

                for xb, yb in train_dl:
                    xb = xb.to(device, non_blocking=True)
                    yb = yb.to(device, non_blocking=True)

                    pred_c = stu(xb)
                    Lc = rmse_only(pred_c, yb)

                    adv_losses = []
                    for eps in PGD_TRAIN_EPS_LIST:
                        xa = pgd_attack(
                            stu, xb, yb,
                            eps=eps,
                            alpha=eps / PGD_ITERS,
                            iters=PGD_ITERS,
                            random_start=PGD_RANDOM_START_TRAIN
                        )
                        pred_a = stu(xa)
                        La = rmse_only(pred_a, yb)
                        adv_losses.append(La)

                    La_mean = torch.stack(adv_losses).mean() if len(adv_losses) > 0 else 0.0 * Lc
                    loss = 0.25 * Lc + 0.75 * La_mean

                    opt.zero_grad()
                    loss.backward()
                    opt.step()

                    bs = xb.size(0)
                    tot += loss.item() * bs
                    cnt += bs

                avg_loss = tot / max(1, cnt)
                ep_mins  = (time.time() - ep_start) / 60.0
                elapsed  = (time.time() - start_time) / 60.0

                history.append({
                    "student_idx": idx,
                    "epoch": ep,
                    "avg_loss": avg_loss,
                    "epoch_time_min": ep_mins,
                    "elapsed_time_min": elapsed,
                })

                print(
                    f"  [PGD Student {idx}] Epoch {ep:02d} | avg_loss={avg_loss:.4f} "
                    f"| epoch={ep_mins:.2f} min | elapsed={elapsed:.2f} min"
                )

        finally:
            monitor.stop()
            stop_heartbeat(hb_flag, hb_thread)

        hist_path = os.path.join(PGD_LOGS_DIR, f"student_{idx}_pgd_train_log.csv")
        pd.DataFrame(history).to_csv(hist_path, index=False)

        outp = os.path.join(PGD_STUDENTS_ADV_DIR, f"student_{idx}_pgd_adv.pth")
        torch.save(stu.to("cpu").state_dict(), outp)
        adv_paths.append(outp)
        print(f"  [PGD Student {idx}] Saved adv student:", outp)

        summary_row = monitor.summary()
        summary_row.update({
            "student": f"student_{idx}",
            "attack_train_type": "pgd",
            "epochs": EPOCHS_STU,
            "learning_rate": LR_STUDENT,
            "iters": PGD_ITERS,
            "total_elapsed_minutes": (time.time() - start_time) / 60.0,
        })
        append_summary_row(summary_row, PGD_TRAIN_RESOURCE_CSV)

    return sorted(adv_paths)

In [ ]:
# =========================================================
# SEAMLESS ADV TRAINING: FGSM -> BIM -> PGD
# =========================================================

fgsm_adv_paths = adv_train_students_fgsm(fgsm_student_paths)
bim_adv_paths  = adv_train_students_bim(bim_student_paths)
pgd_adv_paths  = adv_train_students_pgd(pgd_student_paths)


Training FGSM adv students

[FGSM Student 01] adversarial training...


/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


  [FGSM Student 01] Epoch 01 | avg_loss=0.1177 | epoch=1.04 min | elapsed=1.04 min
  [FGSM Student 01] Epoch 02 | avg_loss=0.1046 | epoch=1.05 min | elapsed=2.09 min
  [FGSM Student 01] Epoch 03 | avg_loss=0.1020 | epoch=1.04 min | elapsed=3.13 min
  [FGSM Student 01] Epoch 04 | avg_loss=0.0994 | epoch=1.03 min | elapsed=4.16 min
  [FGSM Student 01] Epoch 05 | avg_loss=0.0972 | epoch=1.07 min | elapsed=5.23 min
  [FGSM Student 01] Saved adv student: /content/drive/MyDrive/298/Elec/proj_v3/fgsm/students_adv/student_01_fgsm_adv.pth

[FGSM Student 02] adversarial training...


/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))
/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


  [FGSM Student 02] Epoch 01 | avg_loss=0.1169 | epoch=1.06 min | elapsed=1.06 min
  [FGSM Student 02] Epoch 02 | avg_loss=0.1043 | epoch=1.04 min | elapsed=2.10 min
  [FGSM Student 02] Epoch 03 | avg_loss=0.1016 | epoch=1.04 min | elapsed=3.14 min
  [FGSM Student 02] Epoch 04 | avg_loss=0.0998 | epoch=1.04 min | elapsed=4.19 min
  [FGSM Student 02] Epoch 05 | avg_loss=0.0971 | epoch=1.03 min | elapsed=5.22 min
  [FGSM Student 02] Saved adv student: /content/drive/MyDrive/298/Elec/proj_v3/fgsm/students_adv/student_02_fgsm_adv.pth

[FGSM Student 03] adversarial training...


...

/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))
/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


  [PGD Student 03] Epoch 01 | avg_loss=0.1467 | epoch=5.76 min | elapsed=5.76 min
[heartbeat:student_03_pgd_train] running... elapsed=10.00 min
  [PGD Student 03] Epoch 02 | avg_loss=0.1452 | epoch=5.84 min | elapsed=11.60 min
  [PGD Student 03] Epoch 03 | avg_loss=0.1439 | epoch=5.78 min | elapsed=17.39 min
[heartbeat:student_03_pgd_train] running... elapsed=20.00 min
  [PGD Student 03] Epoch 04 | avg_loss=0.1437 | epoch=5.79 min | elapsed=23.17 min
  [PGD Student 03] Epoch 05 | avg_loss=0.1429 | epoch=5.80 min | elapsed=28.97 min
  [PGD Student 03] Saved adv student: /content/drive/MyDrive/298/Elec/proj_v3/pgd/students_adv/student_03_pgd_adv.pth

[PGD Student 04] adversarial training...


/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))
/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


  [PGD Student 04] Epoch 01 | avg_loss=0.1466 | epoch=5.77 min | elapsed=5.77 min
[heartbeat:student_04_pgd_train] running... elapsed=10.00 min
  [PGD Student 04] Epoch 02 | avg_loss=0.1450 | epoch=5.80 min | elapsed=11.57 min
  [PGD Student 04] Epoch 03 | avg_loss=0.1442 | epoch=5.78 min | elapsed=17.34 min
[heartbeat:student_04_pgd_train] running... elapsed=20.00 min
  [PGD Student 04] Epoch 04 | avg_loss=0.1435 | epoch=5.79 min | elapsed=23.13 min
  [PGD Student 04] Epoch 05 | avg_loss=0.1424 | epoch=5.82 min | elapsed=28.95 min
  [PGD Student 04] Saved adv student: /content/drive/MyDrive/298/Elec/proj_v3/pgd/students_adv/student_04_pgd_adv.pth


/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
# =========================================================
# CLEAN EVAL TABLE HELPERS
# =========================================================

def save_clean_eval(base_model, adv_paths, out_csv):
    clean_rows = []

    base_val_rmse = eval_one_epoch(base_model, val_dl)
    base_test_rmse = eval_one_epoch(base_model, test_dl)

    clean_rows.append({
        "model": "base",
        "val_rmse": base_val_rmse,
        "test_rmse": base_test_rmse
    })

    models, names = load_student_models(adv_paths)

    for name, stu in zip(names, models):
        val_rmse = eval_one_epoch(stu, val_dl)
        test_rmse = eval_one_epoch(stu, test_dl)

        clean_rows.append({
            "model": name,
            "val_rmse": val_rmse,
            "test_rmse": test_rmse
        })

        print(f"[{name}] val_rmse={val_rmse:.5f} | test_rmse={test_rmse:.5f}")

    df_clean_eval = pd.DataFrame(clean_rows).sort_values("model").reset_index(drop=True)
    df_clean_eval.to_csv(out_csv, index=False)
    print("Saved clean eval CSV to:", out_csv)

    return df_clean_eval

In [ ]:
# =========================================================
# CLEAN EVALS
# =========================================================

print("\nFGSM clean eval")
df_fgsm_clean = save_clean_eval(best_base, fgsm_adv_paths, FGSM_CLEAN_EVAL_CSV)

print("\nBIM clean eval")
df_bim_clean = save_clean_eval(best_base, bim_adv_paths, BIM_CLEAN_EVAL_CSV)

print("\nPGD clean eval")
df_pgd_clean = save_clean_eval(best_base, pgd_adv_paths, PGD_CLEAN_EVAL_CSV)


FGSM clean eval


/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[student_01_fgsm_adv] val_rmse=0.11619 | test_rmse=0.11359
[student_02_fgsm_adv] val_rmse=0.13375 | test_rmse=0.12433
[student_03_fgsm_adv] val_rmse=0.11257 | test_rmse=0.11062
[student_04_fgsm_adv] val_rmse=0.12305 | test_rmse=0.11830
Saved clean eval CSV to: /content/drive/MyDrive/298/Elec/proj_v3/fgsm/results/students_clean_eval.csv

BIM clean eval


/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[student_01_bim_adv] val_rmse=0.29586 | test_rmse=0.22861
[student_02_bim_adv] val_rmse=0.32959 | test_rmse=0.24789
[student_03_bim_adv] val_rmse=0.25811 | test_rmse=0.21236
[student_04_bim_adv] val_rmse=0.30797 | test_rmse=0.23507
Saved clean eval CSV to: /content/drive/MyDrive/298/Elec/proj_v3/bim/results/students_clean_eval.csv

PGD clean eval


/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[student_01_pgd_adv] val_rmse=0.24124 | test_rmse=0.18973
[student_02_pgd_adv] val_rmse=0.28483 | test_rmse=0.21741
[student_03_pgd_adv] val_rmse=0.29828 | test_rmse=0.22650
[student_04_pgd_adv] val_rmse=0.28934 | test_rmse=0.22066
Saved clean eval CSV to: /content/drive/MyDrive/298/Elec/proj_v3/pgd/results/students_clean_eval.csv


In [ ]:
# =========================================================
# DIVERSITY HELPERS
# =========================================================

def flatten_params(model):
    return torch.cat([p.detach().cpu().view(-1) for p in model.parameters()])

def compute_weight_diversity(students, names, out_csv):
    vecs = [flatten_params(m) for m in students]
    rows = []

    for i, j in itertools.combinations(range(len(vecs)), 2):
        d = torch.norm(vecs[i] - vecs[j], p=2).item()
        rows.append({
            "student_i": names[i],
            "student_j": names[j],
            "l2_distance": d,
        })

    df_w = pd.DataFrame(rows)
    df_w.to_csv(out_csv, index=False)
    print("Saved weight diversity to:", out_csv)
    return df_w

@torch.no_grad()
def compute_prediction_diversity(students, loader, out_csv):
    for m in students:
        m.eval()

    H = HZN_STEPS
    sum_pred  = np.zeros(H, dtype=np.float64)
    sum_pred2 = np.zeros(H, dtype=np.float64)
    count     = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        preds_stu = []

        for m in students:
            preds_stu.append(m(xb).cpu().numpy())

        preds_stu = np.stack(preds_stu, axis=0)
        preds_mean_batch = preds_stu.mean(axis=1).squeeze(-1)

        sum_pred  += preds_mean_batch.mean(axis=0)
        sum_pred2 += (preds_mean_batch ** 2).mean(axis=0)
        count += 1

    mean_pred  = sum_pred / max(1, count)
    mean_pred2 = sum_pred2 / max(1, count)
    var_pred   = np.maximum(mean_pred2 - mean_pred**2, 0.0)

    df_p = pd.DataFrame({
        "horizon_idx": np.arange(H),
        "mean_pred": mean_pred,
        "var_pred": var_pred,
    })
    df_p.to_csv(out_csv, index=False)
    print("Saved prediction diversity to:", out_csv)
    return df_p

In [ ]:
# =========================================================
# SINGLE-PASS EVAL HELPERS
# =========================================================

def compute_transfer_matrix(
    students,
    names,
    loader,
    attack_fn,
    eps_list,
    extra_kwargs_func,
    out_csv,
    trace_csv,
    summary_csv,
    attack_label="attack",
):
    rows = []

    hb_flag, hb_thread = start_heartbeat(f"{attack_label}_transfer_matrix")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_time = time.time()

    monitor.start()

    try:
        for eps in eps_list:
            atk_kwargs = extra_kwargs_func(eps)
            print(f"\n[{attack_label.upper()}] Transfer matrix for eps={eps}...")

            for i, atk_name in enumerate(names):
                for j, def_name in enumerate(names):
                    atk_model = students[i]
                    def_model = students[j]

                    rmse_ij = eval_pair_attack(
                        defender=def_model,
                        attacker=atk_model,
                        loader=loader,
                        attack_fn=attack_fn,
                        atk_kwargs=atk_kwargs
                    )

                    rows.append({
                        "epsilon": eps,
                        "attacker": atk_name,
                        "defender": def_name,
                        "rmse_adv": rmse_ij,
                    })

                    print(
                        f"  atk={atk_name} -> def={def_name} | "
                        f"eps={eps:.2f} | RMSE={rmse_ij:.5f}"
                    )
    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    df_xfer = pd.DataFrame(rows)
    df_xfer.to_csv(out_csv, index=False)
    print("Saved transfer matrix to:", out_csv)

    save_monitor_outputs(
        monitor,
        trace_csv=trace_csv,
        summary_csv=summary_csv,
        extra_summary={
            "stage": f"{attack_label}_transfer_matrix",
            "total_elapsed_minutes": (time.time() - start_time) / 60.0,
        }
    )

    return df_xfer

def run_morphence_eval_single_pass(
    attack_name,
    attack_fn,
    eps_list,
    students,
    results_csv,
    stats_csv,
    extra_kwargs_func,
    trace_csv,
    summary_csv,
):
    all_rows = []

    hb_flag, hb_thread = start_heartbeat(f"{attack_name}_single_pass_eval")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_all = time.time()

    monitor.start()

    try:
        base_clean = eval_clean_rmse(best_base, test_dl)
        morph_clean = eval_ensemble_rmse(students, test_dl)

        all_rows.append(["clean", None, "base", base_clean, 1, SEED])
        all_rows.append(["clean", None, "morph_ensemble", morph_clean, 1, SEED])

        print(
            f"[{attack_name.upper()}] clean | "
            f"base={base_clean:.5f} | morph_ensemble={morph_clean:.5f}"
        )

        for eps in eps_list:
            atk_kwargs = extra_kwargs_func(eps)

            base_rmse = eval_pair_attack(
                defender=best_base,
                attacker=best_base,
                loader=test_dl,
                attack_fn=attack_fn,
                atk_kwargs=atk_kwargs
            )

            def atk_kwargs_wrapper():
                return atk_kwargs

            morph_rmse = eval_random_switch_attack(
                models=students,
                loader=test_dl,
                attack_fn=attack_fn,
                atk_kwargs_func=atk_kwargs_wrapper
            )

            all_rows.append([attack_name, eps, "base", base_rmse, 1, SEED])
            all_rows.append([attack_name, eps, "morph_rand", morph_rmse, 1, SEED])

            print(
                f"[{attack_name.upper()}] eps={eps:.2f} | "
                f"base={base_rmse:.5f} | morph_rand={morph_rmse:.5f}"
            )

    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    df_runs = pd.DataFrame(
        all_rows,
        columns=["attack", "epsilon", "model", "RMSE", "run", "seed"]
    )
    df_runs.to_csv(results_csv, index=False)

    stats = (
        df_runs
        .groupby(["attack", "epsilon", "model"])["RMSE"]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
        .sort_values(["attack", "epsilon", "model"])
    )
    stats.to_csv(stats_csv, index=False)

    save_monitor_outputs(
        monitor,
        trace_csv=trace_csv,
        summary_csv=summary_csv,
        extra_summary={
            "stage": f"{attack_name}_single_pass_eval",
            "total_elapsed_minutes": (time.time() - start_all) / 60.0,
        }
    )

    print("Saved single-pass eval CSV to:", results_csv)
    print("Saved single-pass stats CSV to:", stats_csv)

    return df_runs, stats

In [ ]:
# =========================================================
# LOAD STUDENT POOLS
# =========================================================

fgsm_students, fgsm_student_names = load_student_models(fgsm_adv_paths)
bim_students,  bim_student_names  = load_student_models(bim_adv_paths)
pgd_students,  pgd_student_names  = load_student_models(pgd_adv_paths)

print("Loaded FGSM students:", len(fgsm_students))
print("Loaded BIM students :", len(bim_students))
print("Loaded PGD students :", len(pgd_students))

/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Loaded FGSM students: 4
Loaded BIM students : 4
Loaded PGD students : 4


In [ ]:
# =========================================================
# FGSM: DIVERSITY + TRANSFER + SINGLE-PASS EVAL
# =========================================================

def fgsm_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps * RFGSM_ALPHA_FACTOR,
    }

print("\nFGSM diversity + transferability + single-pass eval")

_ = compute_weight_diversity(fgsm_students, fgsm_student_names, FGSM_WEIGHT_DIVERSITY_CSV)
_ = compute_prediction_diversity(fgsm_students, test_dl, FGSM_PRED_DIVERSITY_CSV)

_ = compute_transfer_matrix(
    students=fgsm_students,
    names=fgsm_student_names,
    loader=test_dl,
    attack_fn=rfgsm_attack,
    eps_list=FGSM_EVAL_EPS_LIST,
    extra_kwargs_func=fgsm_kwargs,
    out_csv=FGSM_XFER_MATRIX_CSV,
    trace_csv=FGSM_XFER_TRACE_CSV,
    summary_csv=FGSM_XFER_SUMMARY_CSV,
    attack_label="fgsm",
)

df_fgsm_runs, fgsm_stats = run_morphence_eval_single_pass(
    attack_name="rfgsm",
    attack_fn=rfgsm_attack,
    eps_list=FGSM_EVAL_EPS_LIST,
    students=fgsm_students,
    results_csv=FGSM_SINGLE_RUN_CSV,
    stats_csv=FGSM_STATS_CSV,
    extra_kwargs_func=fgsm_kwargs,
    trace_csv=FGSM_EVAL_TRACE_CSV,
    summary_csv=FGSM_EVAL_SUMMARY_CSV,
)


FGSM diversity + transferability + single-pass eval
Saved weight diversity to: /content/drive/MyDrive/298/Elec/proj_v3/fgsm/results/fgsm_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Elec/proj_v3/fgsm/results/fgsm_prediction_diversity.csv

[FGSM] Transfer matrix for eps=0.1...
  atk=student_01_fgsm_adv -> def=student_01_fgsm_adv | eps=0.10 | RMSE=0.12593
  atk=student_01_fgsm_adv -> def=student_02_fgsm_adv | eps=0.10 | RMSE=0.12758
  atk=student_01_fgsm_adv -> def=student_03_fgsm_adv | eps=0.10 | RMSE=0.11653
  atk=student_01_fgsm_adv -> def=student_04_fgsm_adv | eps=0.10 | RMSE=0.12074
  atk=student_02_fgsm_adv -> def=student_01_fgsm_adv | eps=0.10 | RMSE=0.11969
  atk=student_02_fgsm_adv -> def=student_02_fgsm_adv | eps=0.10 | RMSE=0.13602
  atk=student_02_fgsm_adv -> def=student_03_fgsm_adv | eps=0.10 | RMSE=0.11761
  atk=student_02_fgsm_adv -> def=student_04_fgsm_adv | eps=0.10 | RMSE=0.11940
  atk=student_03_fgsm_adv -> def=student_01_fgsm_adv | e

/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


[RFGSM] clean | base=0.11526 | morph_ensemble=0.11355
[RFGSM] eps=0.10 | base=0.17688 | morph_rand=0.12076
[RFGSM] eps=0.20 | base=0.24146 | morph_rand=0.11717
[RFGSM] eps=0.30 | base=0.27625 | morph_rand=0.10711
[RFGSM] eps=0.50 | base=0.31656 | morph_rand=0.10513
Saved resource trace to: /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/fgsm_eval_resource_trace.csv
Saved resource summary to: /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/fgsm_eval_resource_summary.csv
Resource summary: {'n_samples': 12, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 1807.0078125, 'max_gpu_mem_mb': 1333.0, 'avg_gpu_util_percent': 30.583333333333332, 'gpu_active_percent_of_samples': 91.66666666666666, 'est_gpu_energy_j': 3600.4372721028326, 'stage': 'rfgsm_single_pass_eval', 'total_elapsed_minutes': 0.42043952147165936}
Saved single-pass eval CSV to: /content/drive/MyDrive/298/Elec/proj_v3/fgsm/results/fgsm_morphence_single_run.csv
Saved single-pass stats CSV to: /content/driv

/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
# =========================================================
# BIM: DIVERSITY + TRANSFER + SINGLE-PASS EVAL
# =========================================================

def bim_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps / BIM_ITERS,
        "iters": BIM_ITERS,
        "random_start": BIM_RANDOM_START_EVAL,
    }

print("\nBIM diversity + transferability + single-pass eval")

_ = compute_weight_diversity(bim_students, bim_student_names, BIM_WEIGHT_DIVERSITY_CSV)
_ = compute_prediction_diversity(bim_students, test_dl, BIM_PRED_DIVERSITY_CSV)

_ = compute_transfer_matrix(
    students=bim_students,
    names=bim_student_names,
    loader=test_dl,
    attack_fn=bim_attack,
    eps_list=BIM_EVAL_EPS_LIST,
    extra_kwargs_func=bim_kwargs,
    out_csv=BIM_XFER_MATRIX_CSV,
    trace_csv=BIM_XFER_TRACE_CSV,
    summary_csv=BIM_XFER_SUMMARY_CSV,
    attack_label="bim",
)

df_bim_runs, bim_stats = run_morphence_eval_single_pass(
    attack_name="bim",
    attack_fn=bim_attack,
    eps_list=BIM_EVAL_EPS_LIST,
    students=bim_students,
    results_csv=BIM_SINGLE_RUN_CSV,
    stats_csv=BIM_STATS_CSV,
    extra_kwargs_func=bim_kwargs,
    trace_csv=BIM_EVAL_TRACE_CSV,
    summary_csv=BIM_EVAL_SUMMARY_CSV,
)


BIM diversity + transferability + single-pass eval
Saved weight diversity to: /content/drive/MyDrive/298/Elec/proj_v3/bim/results/bim_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Elec/proj_v3/bim/results/bim_prediction_diversity.csv

[BIM] Transfer matrix for eps=0.1...
  atk=student_01_bim_adv -> def=student_01_bim_adv | eps=0.10 | RMSE=0.22864
  atk=student_01_bim_adv -> def=student_02_bim_adv | eps=0.10 | RMSE=0.24789
  atk=student_01_bim_adv -> def=student_03_bim_adv | eps=0.10 | RMSE=0.21242
  atk=student_01_bim_adv -> def=student_04_bim_adv | eps=0.10 | RMSE=0.23509
  atk=student_02_bim_adv -> def=student_01_bim_adv | eps=0.10 | RMSE=0.22862
  atk=student_02_bim_adv -> def=student_02_bim_adv | eps=0.10 | RMSE=0.24790
  atk=student_02_bim_adv -> def=student_03_bim_adv | eps=0.10 | RMSE=0.21239
  atk=student_02_bim_adv -> def=student_04_bim_adv | eps=0.10 | RMSE=0.23508
  atk=student_03_bim_adv -> def=student_01_bim_adv | eps=0.10 | RMSE=0.22863
 

/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


[BIM] clean | base=0.11526 | morph_ensemble=0.22948
[BIM] eps=0.10 | base=0.16996 | morph_rand=0.23274
[BIM] eps=0.20 | base=0.25715 | morph_rand=0.23276
[BIM] eps=0.30 | base=0.32315 | morph_rand=0.23337
[BIM] eps=0.50 | base=0.38540 | morph_rand=0.23016
Saved resource trace to: /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/bim_eval_resource_trace.csv
Saved resource summary to: /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/bim_eval_resource_summary.csv
Resource summary: {'n_samples': 92, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 1800.4921875, 'max_gpu_mem_mb': 1411.0, 'avg_gpu_util_percent': 46.858695652173914, 'gpu_active_percent_of_samples': 98.91304347826086, 'est_gpu_energy_j': 31284.363154028655, 'stage': 'bim_single_pass_eval', 'total_elapsed_minutes': 3.109342094262441}
Saved single-pass eval CSV to: /content/drive/MyDrive/298/Elec/proj_v3/bim/results/bim_morphence_single_run.csv
Saved single-pass stats CSV to: /content/drive/MyDrive/298/Elec

/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
# =========================================================
# PGD: DIVERSITY + TRANSFER + SINGLE-PASS EVAL
# =========================================================

def pgd_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps / PGD_ITERS,
        "iters": PGD_ITERS,
        "random_start": PGD_RANDOM_START_EVAL,
    }

print("\nPGD diversity + transferability + single-pass eval")

_ = compute_weight_diversity(pgd_students, pgd_student_names, PGD_WEIGHT_DIVERSITY_CSV)
_ = compute_prediction_diversity(pgd_students, test_dl, PGD_PRED_DIVERSITY_CSV)

_ = compute_transfer_matrix(
    students=pgd_students,
    names=pgd_student_names,
    loader=test_dl,
    attack_fn=pgd_attack,
    eps_list=PGD_EVAL_EPS_LIST,
    extra_kwargs_func=pgd_kwargs,
    out_csv=PGD_XFER_MATRIX_CSV,
    trace_csv=PGD_XFER_TRACE_CSV,
    summary_csv=PGD_XFER_SUMMARY_CSV,
    attack_label="pgd",
)

df_pgd_runs, pgd_stats = run_morphence_eval_single_pass(
    attack_name="pgd",
    attack_fn=pgd_attack,
    eps_list=PGD_EVAL_EPS_LIST,
    students=pgd_students,
    results_csv=PGD_SINGLE_RUN_CSV,
    stats_csv=PGD_STATS_CSV,
    extra_kwargs_func=pgd_kwargs,
    trace_csv=PGD_EVAL_TRACE_CSV,
    summary_csv=PGD_EVAL_SUMMARY_CSV,
)


PGD diversity + transferability + single-pass eval
Saved weight diversity to: /content/drive/MyDrive/298/Elec/proj_v3/pgd/results/pgd_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Elec/proj_v3/pgd/results/pgd_prediction_diversity.csv

[PGD] Transfer matrix for eps=0.1...
  atk=student_01_pgd_adv -> def=student_01_pgd_adv | eps=0.10 | RMSE=0.19182
  atk=student_01_pgd_adv -> def=student_02_pgd_adv | eps=0.10 | RMSE=0.21920
  atk=student_01_pgd_adv -> def=student_03_pgd_adv | eps=0.10 | RMSE=0.22785
  atk=student_01_pgd_adv -> def=student_04_pgd_adv | eps=0.10 | RMSE=0.22276
  atk=student_02_pgd_adv -> def=student_01_pgd_adv | eps=0.10 | RMSE=0.19244
  atk=student_02_pgd_adv -> def=student_02_pgd_adv | eps=0.10 | RMSE=0.21988
  atk=student_02_pgd_adv -> def=student_03_pgd_adv | eps=0.10 | RMSE=0.22846
  atk=student_02_pgd_adv -> def=student_04_pgd_adv | eps=0.10 | RMSE=0.22335
  atk=student_03_pgd_adv -> def=student_01_pgd_adv | eps=0.10 | RMSE=0.19103
 

/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


[PGD] clean | base=0.11526 | morph_ensemble=0.21315
[PGD] eps=0.10 | base=0.16993 | morph_rand=0.21668
[PGD] eps=0.20 | base=0.25712 | morph_rand=0.22066
[PGD] eps=0.30 | base=0.32243 | morph_rand=0.22543
[PGD] eps=0.50 | base=0.38510 | morph_rand=0.23827
Saved resource trace to: /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/pgd_eval_resource_trace.csv
Saved resource summary to: /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/pgd_eval_resource_summary.csv
Resource summary: {'n_samples': 92, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 1812.09375, 'max_gpu_mem_mb': 1473.0, 'avg_gpu_util_percent': 46.57608695652174, 'gpu_active_percent_of_samples': 98.91304347826086, 'est_gpu_energy_j': 31398.749439936877, 'stage': 'pgd_single_pass_eval', 'total_elapsed_minutes': 3.109448957443237}
Saved single-pass eval CSV to: /content/drive/MyDrive/298/Elec/proj_v3/pgd/results/pgd_morphence_single_run.csv
Saved single-pass stats CSV to: /content/drive/MyDrive/298/Elec/pr

/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
print("\n================ ECL PROJ_V3 OUTPUTS ================\n")

print("Base checkpoint              :", BASE_CKPT_PATH)
print("Base train log               :", BASE_TRAIN_LOG_CSV)
print("Base resource trace          :", BASE_RESOURCE_TRACE_CSV)
print("Base resource summary        :", BASE_RESOURCE_SUMMARY_CSV)
print("Base model profile           :", BASE_PROFILE_CSV)

print("\nFGSM clean eval              :", FGSM_CLEAN_EVAL_CSV)
print("FGSM train resource          :", FGSM_TRAIN_RESOURCE_CSV)
print("FGSM weight diversity        :", FGSM_WEIGHT_DIVERSITY_CSV)
print("FGSM prediction diversity    :", FGSM_PRED_DIVERSITY_CSV)
print("FGSM transfer matrix         :", FGSM_XFER_MATRIX_CSV)
print("FGSM transfer trace          :", FGSM_XFER_TRACE_CSV)
print("FGSM transfer summary        :", FGSM_XFER_SUMMARY_CSV)
print("FGSM single-pass eval        :", FGSM_SINGLE_RUN_CSV)
print("FGSM single-pass stats       :", FGSM_STATS_CSV)
print("FGSM eval trace              :", FGSM_EVAL_TRACE_CSV)
print("FGSM eval summary            :", FGSM_EVAL_SUMMARY_CSV)

print("\nBIM clean eval               :", BIM_CLEAN_EVAL_CSV)
print("BIM train resource           :", BIM_TRAIN_RESOURCE_CSV)
print("BIM weight diversity         :", BIM_WEIGHT_DIVERSITY_CSV)
print("BIM prediction diversity     :", BIM_PRED_DIVERSITY_CSV)
print("BIM transfer matrix          :", BIM_XFER_MATRIX_CSV)
print("BIM transfer trace           :", BIM_XFER_TRACE_CSV)
print("BIM transfer summary         :", BIM_XFER_SUMMARY_CSV)
print("BIM single-pass eval         :", BIM_SINGLE_RUN_CSV)
print("BIM single-pass stats        :", BIM_STATS_CSV)
print("BIM eval trace               :", BIM_EVAL_TRACE_CSV)
print("BIM eval summary             :", BIM_EVAL_SUMMARY_CSV)

print("\nPGD clean eval               :", PGD_CLEAN_EVAL_CSV)
print("PGD train resource           :", PGD_TRAIN_RESOURCE_CSV)
print("PGD weight diversity         :", PGD_WEIGHT_DIVERSITY_CSV)
print("PGD prediction diversity     :", PGD_PRED_DIVERSITY_CSV)
print("PGD transfer matrix          :", PGD_XFER_MATRIX_CSV)
print("PGD transfer trace           :", PGD_XFER_TRACE_CSV)
print("PGD transfer summary         :", PGD_XFER_SUMMARY_CSV)
print("PGD single-pass eval         :", PGD_SINGLE_RUN_CSV)
print("PGD single-pass stats        :", PGD_STATS_CSV)
print("PGD eval trace               :", PGD_EVAL_TRACE_CSV)
print("PGD eval summary             :", PGD_EVAL_SUMMARY_CSV)

print("\nCommon artifacts dir         :", COMMON_ARTIFACTS_DIR)
print("Common logs dir              :", COMMON_LOGS_DIR)
print("Computational metrics dir    :", COMP_METRICS_DIR)


================ ECL PROJ_V3 OUTPUTS ================

Base checkpoint              : /content/drive/MyDrive/298/Elec/proj_v3/baseModel/elec_base_best.pth
Base train log               : /content/drive/MyDrive/298/Elec/proj_v3/common_logs/elec_base_train_log.csv
Base resource trace          : /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/base_resource_trace.csv
Base resource summary        : /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/base_resource_summary.csv
Base model profile           : /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/base_model_profile.csv

FGSM clean eval              : /content/drive/MyDrive/298/Elec/proj_v3/fgsm/results/students_clean_eval.csv
FGSM train resource          : /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/fgsm_student_train_resource_summary.csv
FGSM weight diversity        : /content/drive/MyDrive/298/Elec/proj_v3/fgsm/results/fgsm_weight_diversity.csv
FGSM prediction diversity    : /c

In [ ]:
# =========================================================
# ECL NOVEL HALF PATHS (proj_v3, same notebook continuation)
# =========================================================

NOVEL_DIR = os.path.join(PROJECT_DIR, "novel")

NOVEL_STUDENTS_DIR = os.path.join(NOVEL_DIR, "students_nov")
NOVEL_RESULTS_DIR  = os.path.join(NOVEL_DIR, "results")
NOVEL_ARTIFACTS_DIR = os.path.join(NOVEL_RESULTS_DIR, "artifacts")
NOVEL_MODEL_SIZES_CSV = os.path.join(NOVEL_RESULTS_DIR, "model_sizes_nov.csv")

FGSM_NOV_DIR = os.path.join(NOVEL_DIR, "fgsm")
BIM_NOV_DIR  = os.path.join(NOVEL_DIR, "bim")
PGD_NOV_DIR  = os.path.join(NOVEL_DIR, "pgd")

FGSM_NOV_STUDENTS_ADV_DIR = os.path.join(FGSM_NOV_DIR, "students_adv_nov_fgsm")
FGSM_NOV_RESULTS_DIR      = os.path.join(FGSM_NOV_DIR, "results")
FGSM_NOV_LOGS_DIR         = os.path.join(FGSM_NOV_RESULTS_DIR, "train_logs")

BIM_NOV_STUDENTS_ADV_DIR = os.path.join(BIM_NOV_DIR, "students_adv_nov_bim")
BIM_NOV_RESULTS_DIR      = os.path.join(BIM_NOV_DIR, "results")
BIM_NOV_LOGS_DIR         = os.path.join(BIM_NOV_RESULTS_DIR, "train_logs")

PGD_NOV_STUDENTS_ADV_DIR = os.path.join(PGD_NOV_DIR, "students_adv_nov_pgd")
PGD_NOV_RESULTS_DIR      = os.path.join(PGD_NOV_DIR, "results")
PGD_NOV_LOGS_DIR         = os.path.join(PGD_NOV_RESULTS_DIR, "train_logs")

for d in [
    NOVEL_DIR, NOVEL_STUDENTS_DIR, NOVEL_RESULTS_DIR, NOVEL_ARTIFACTS_DIR,
    FGSM_NOV_DIR, FGSM_NOV_STUDENTS_ADV_DIR, FGSM_NOV_RESULTS_DIR, FGSM_NOV_LOGS_DIR,
    BIM_NOV_DIR,  BIM_NOV_STUDENTS_ADV_DIR,  BIM_NOV_RESULTS_DIR,  BIM_NOV_LOGS_DIR,
    PGD_NOV_DIR,  PGD_NOV_STUDENTS_ADV_DIR,  PGD_NOV_RESULTS_DIR,  PGD_NOV_LOGS_DIR,
]:
    os.makedirs(d, exist_ok=True)

# FGSM novel files
NOV_FGSM_RUNS_CSV         = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_morphence_single_run.csv")
NOV_FGSM_STATS_CSV        = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_morphence_single_pass_stats.csv")
NOV_FGSM_WEIGHT_CSV       = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_weight_diversity.csv")
NOV_FGSM_PRED_CSV         = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_prediction_diversity.csv")
NOV_FGSM_XFER_CSV         = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_transfer_matrix.csv")
NOV_FGSM_CLEAN_EVAL_CSV   = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_clean_eval.csv")

# BIM novel files
NOV_BIM_RUNS_CSV          = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_morphence_single_run.csv")
NOV_BIM_STATS_CSV         = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_morphence_single_pass_stats.csv")
NOV_BIM_WEIGHT_CSV        = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_weight_diversity.csv")
NOV_BIM_PRED_CSV          = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_prediction_diversity.csv")
NOV_BIM_XFER_CSV          = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_transfer_matrix.csv")
NOV_BIM_CLEAN_EVAL_CSV    = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_clean_eval.csv")

# PGD novel files
NOV_PGD_RUNS_CSV          = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_morphence_single_run.csv")
NOV_PGD_STATS_CSV         = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_morphence_single_pass_stats.csv")
NOV_PGD_WEIGHT_CSV        = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_weight_diversity.csv")
NOV_PGD_PRED_CSV          = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_prediction_diversity.csv")
NOV_PGD_XFER_CSV          = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_transfer_matrix.csv")
NOV_PGD_CLEAN_EVAL_CSV    = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_clean_eval.csv")

# computational metrics
NOVEL_COMP_DIR = os.path.join(NOVEL_DIR, "computational_metrics")
os.makedirs(NOVEL_COMP_DIR, exist_ok=True)

FGSM_NOV_TRAIN_RESOURCE_CSV = os.path.join(NOVEL_COMP_DIR, "fgsm_nov_student_train_resource_summary.csv")
BIM_NOV_TRAIN_RESOURCE_CSV  = os.path.join(NOVEL_COMP_DIR, "bim_nov_student_train_resource_summary.csv")
PGD_NOV_TRAIN_RESOURCE_CSV  = os.path.join(NOVEL_COMP_DIR, "pgd_nov_student_train_resource_summary.csv")

FGSM_NOV_XFER_TRACE_CSV   = os.path.join(NOVEL_COMP_DIR, "fgsm_nov_transfer_resource_trace.csv")
FGSM_NOV_XFER_SUMMARY_CSV = os.path.join(NOVEL_COMP_DIR, "fgsm_nov_transfer_resource_summary.csv")
BIM_NOV_XFER_TRACE_CSV    = os.path.join(NOVEL_COMP_DIR, "bim_nov_transfer_resource_trace.csv")
BIM_NOV_XFER_SUMMARY_CSV  = os.path.join(NOVEL_COMP_DIR, "bim_nov_transfer_resource_summary.csv")
PGD_NOV_XFER_TRACE_CSV    = os.path.join(NOVEL_COMP_DIR, "pgd_nov_transfer_resource_trace.csv")
PGD_NOV_XFER_SUMMARY_CSV  = os.path.join(NOVEL_COMP_DIR, "pgd_nov_transfer_resource_summary.csv")

FGSM_NOV_EVAL_TRACE_CSV   = os.path.join(NOVEL_COMP_DIR, "fgsm_nov_eval_resource_trace.csv")
FGSM_NOV_EVAL_SUMMARY_CSV = os.path.join(NOVEL_COMP_DIR, "fgsm_nov_eval_resource_summary.csv")
BIM_NOV_EVAL_TRACE_CSV    = os.path.join(NOVEL_COMP_DIR, "bim_nov_eval_resource_trace.csv")
BIM_NOV_EVAL_SUMMARY_CSV  = os.path.join(NOVEL_COMP_DIR, "bim_nov_eval_resource_summary.csv")
PGD_NOV_EVAL_TRACE_CSV    = os.path.join(NOVEL_COMP_DIR, "pgd_nov_eval_resource_trace.csv")
PGD_NOV_EVAL_SUMMARY_CSV  = os.path.join(NOVEL_COMP_DIR, "pgd_nov_eval_resource_summary.csv")

print("NOVEL_DIR:", NOVEL_DIR)
print("NOVEL_COMP_DIR:", NOVEL_COMP_DIR)

NOVEL_DIR: /content/drive/MyDrive/298/Elec/proj_v3/novel
NOVEL_COMP_DIR: /content/drive/MyDrive/298/Elec/proj_v3/novel/computational_metrics


In [ ]:
# =========================================================
# LOAD BASE MODEL FOR NOVEL STUDENTS
# =========================================================

assert os.path.exists(BASE_CKPT_PATH), f"Base checkpoint not found: {BASE_CKPT_PATH}"

best_base = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to(device)
best_base.load_state_dict(torch.load(BASE_CKPT_PATH, map_location=device))
best_base.eval()

base_cpu = ElecSeq2SeqTransformer(
    d_feats=D_FEATS,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=DIM_FF,
    dropout=DROPOUT,
    hist_steps=HIST_STEPS,
    hzn_steps=HZN_STEPS,
    patch=PATCH
).to("cpu")
base_cpu.load_state_dict(torch.load(BASE_CKPT_PATH, map_location="cpu"))
base_cpu.eval()

print("Loaded base model for novel student generation from:", BASE_CKPT_PATH)

Loaded base model for novel student generation from: /content/drive/MyDrive/298/Elec/proj_v3/baseModel/elec_base_best.pth


/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


In [ ]:
# =========================================================
# NOVEL STUDENT GENERATION
# =========================================================

def model_n_params(model: nn.Module):
    return sum(p.numel() for p in model.parameters())

def model_file_size_mb(path: str):
    if not os.path.exists(path):
        return None
    return os.path.getsize(path) / (1024.0 * 1024.0)

def append_df_to_csv(df, path):
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=header, index=False)

def is_enc_self_attn_param(name: str) -> bool:
    return name.startswith("encoder.layers.") and ".self_attn." in name

def is_dec_attn_param(name: str) -> bool:
    if not name.startswith("decoder.layers."):
        return False
    return (".self_attn." in name) or (".multihead_attn." in name)

def is_ffn_param(name: str) -> bool:
    if not (name.startswith("encoder.layers.") or name.startswith("decoder.layers.")):
        return False
    return (".linear1." in name) or (".linear2." in name)

def is_norm_param(name: str) -> bool:
    if not (name.startswith("encoder.layers.") or name.startswith("decoder.layers.")):
        return False
    return ".norm" in name

novel_student_specs = [
    dict(idx=5, name="student_05_enc_attn", selector=is_enc_self_attn_param, noise_scale=1e-3),
    dict(idx=6, name="student_06_dec_attn", selector=is_dec_attn_param,      noise_scale=1e-3),
    dict(idx=7, name="student_07_ffn",      selector=is_ffn_param,           noise_scale=1e-3),
    dict(idx=8, name="student_08_norms",    selector=is_norm_param,          noise_scale=1e-3),
]

novel_student_paths = []

print("\n[Novel] creating students 5–8 from base")

for spec in novel_student_specs:
    idx      = spec["idx"]
    name     = spec["name"]
    selector = spec["selector"]
    sigma    = spec["noise_scale"]

    st = copy.deepcopy(base_cpu)

    with torch.no_grad():
        for pname, p in st.named_parameters():
            if selector(pname):
                p.add_(sigma * torch.randn_like(p))

    outp = os.path.join(NOVEL_STUDENTS_DIR, f"{name}.pth")
    torch.save(st.state_dict(), outp)
    novel_student_paths.append((idx, name, outp))
    print(f"  saved novel student {idx}: {outp}")

    size_mb = model_file_size_mb(outp)
    df_ms = pd.DataFrame([{
        "student_idx": idx,
        "model_name": name,
        "path": outp,
        "params": model_n_params(base_cpu),
        "size_mb": size_mb,
    }])
    append_df_to_csv(df_ms, NOVEL_MODEL_SIZES_CSV)

novel_student_paths = sorted(novel_student_paths, key=lambda x: x[0])

print("\nNovel students created:")
for idx, name, path in novel_student_paths:
    print(idx, name, path)


[Novel] creating students 5–8 from base
  saved novel student 5: /content/drive/MyDrive/298/Elec/proj_v3/novel/students_nov/student_05_enc_attn.pth
  saved novel student 6: /content/drive/MyDrive/298/Elec/proj_v3/novel/students_nov/student_06_dec_attn.pth
  saved novel student 7: /content/drive/MyDrive/298/Elec/proj_v3/novel/students_nov/student_07_ffn.pth
  saved novel student 8: /content/drive/MyDrive/298/Elec/proj_v3/novel/students_nov/student_08_norms.pth

Novel students created:
5 student_05_enc_attn /content/drive/MyDrive/298/Elec/proj_v3/novel/students_nov/student_05_enc_attn.pth
6 student_06_dec_attn /content/drive/MyDrive/298/Elec/proj_v3/novel/students_nov/student_06_dec_attn.pth
7 student_07_ffn /content/drive/MyDrive/298/Elec/proj_v3/novel/students_nov/student_07_ffn.pth
8 student_08_norms /content/drive/MyDrive/298/Elec/proj_v3/novel/students_nov/student_08_norms.pth


In [ ]:
# =========================================================
# NOVEL ADV TRAINING HELPERS
# =========================================================

def adv_train_novel_students_fgsm(novel_student_paths):
    adv_paths = []

    print("\n[Novel] Training FGSM adv novel students")
    for idx, name, sp in novel_student_paths:
        stu = ElecSeq2SeqTransformer(
            d_feats=D_FEATS,
            d_model=D_MODEL,
            nhead=N_HEAD,
            num_layers=N_LAYERS,
            dim_ff=DIM_FF,
            dropout=DROPOUT,
            hist_steps=HIST_STEPS,
            hzn_steps=HZN_STEPS,
            patch=PATCH
        ).to(device)
        stu.load_state_dict(torch.load(sp, map_location=device))

        opt = optim.AdamW(stu.parameters(), lr=LR_STUDENT)
        history = []

        hb_flag, hb_thread = start_heartbeat(f"{name}_fgsm_train")
        monitor = ResourceMonitor(sample_interval=2.0)
        start_time = time.time()
        monitor.start()

        try:
            for ep in range(1, EPOCHS_STU + 1):
                stu.train()
                tot = cnt = 0
                ep_start = time.time()

                for xb, yb in train_dl:
                    xb = xb.to(device, non_blocking=True)
                    yb = yb.to(device, non_blocking=True)

                    pred_c = stu(xb)
                    Lc = rmse_only(pred_c, yb)

                    adv_losses = []
                    for eps in FGSM_TRAIN_EPS_LIST:
                        xa = rfgsm_attack(stu, xb, yb, eps=eps)
                        pred_a = stu(xa)
                        La = rmse_only(pred_a, yb)
                        adv_losses.append(La)

                    La_mean = torch.stack(adv_losses).mean() if len(adv_losses) > 0 else 0.0 * Lc
                    loss = 0.25 * Lc + 0.75 * La_mean

                    opt.zero_grad()
                    loss.backward()
                    opt.step()

                    bs = xb.size(0)
                    tot += loss.item() * bs
                    cnt += bs

                avg_loss = tot / max(1, cnt)
                ep_mins  = (time.time() - ep_start) / 60.0
                elapsed  = (time.time() - start_time) / 60.0

                history.append({
                    "student_idx": idx,
                    "student_name": name,
                    "epoch": ep,
                    "avg_loss": avg_loss,
                    "epoch_time_min": ep_mins,
                    "elapsed_time_min": elapsed,
                })

                print(f"[FGSM-NOV {name}] Epoch {ep:02d} | avg_loss={avg_loss:.4f} | epoch={ep_mins:.2f} min | elapsed={elapsed:.2f} min")

        finally:
            monitor.stop()
            stop_heartbeat(hb_flag, hb_thread)

        hist_path = os.path.join(FGSM_NOV_LOGS_DIR, f"{name}_fgsm_train_history.csv")
        pd.DataFrame(history).to_csv(hist_path, index=False)

        outp = os.path.join(FGSM_NOV_STUDENTS_ADV_DIR, f"{name}_fgsm_adv.pth")
        torch.save(stu.to("cpu").state_dict(), outp)
        adv_paths.append((idx, name, outp))
        print("Saved FGSM novel adv student:", outp)

        summary_row = monitor.summary()
        summary_row.update({
            "student": name,
            "attack_train_type": "fgsm_nov",
            "epochs": EPOCHS_STU,
            "learning_rate": LR_STUDENT,
            "total_elapsed_minutes": (time.time() - start_time) / 60.0,
        })
        append_summary_row(summary_row, FGSM_NOV_TRAIN_RESOURCE_CSV)

    return sorted(adv_paths, key=lambda x: x[0])


def adv_train_novel_students_bim(novel_student_paths):
    adv_paths = []

    print("\n[Novel] Training BIM adv novel students")
    for idx, name, sp in novel_student_paths:
        stu = ElecSeq2SeqTransformer(
            d_feats=D_FEATS,
            d_model=D_MODEL,
            nhead=N_HEAD,
            num_layers=N_LAYERS,
            dim_ff=DIM_FF,
            dropout=DROPOUT,
            hist_steps=HIST_STEPS,
            hzn_steps=HZN_STEPS,
            patch=PATCH
        ).to(device)
        stu.load_state_dict(torch.load(sp, map_location=device))

        opt = optim.AdamW(stu.parameters(), lr=LR_STUDENT)
        history = []

        hb_flag, hb_thread = start_heartbeat(f"{name}_bim_train")
        monitor = ResourceMonitor(sample_interval=2.0)
        start_time = time.time()
        monitor.start()

        try:
            for ep in range(1, EPOCHS_STU + 1):
                stu.train()
                tot = cnt = 0
                ep_start = time.time()

                for xb, yb in train_dl:
                    xb = xb.to(device, non_blocking=True)
                    yb = yb.to(device, non_blocking=True)

                    pred_c = stu(xb)
                    Lc = rmse_only(pred_c, yb)

                    adv_losses = []
                    for eps in BIM_TRAIN_EPS_LIST:
                        xa = bim_attack(
                            stu, xb, yb,
                            eps=eps,
                            alpha=eps / BIM_ITERS,
                            iters=BIM_ITERS,
                            random_start=BIM_RANDOM_START_TRAIN
                        )
                        pred_a = stu(xa)
                        La = rmse_only(pred_a, yb)
                        adv_losses.append(La)

                    La_mean = torch.stack(adv_losses).mean() if len(adv_losses) > 0 else 0.0 * Lc
                    loss = 0.25 * Lc + 0.75 * La_mean

                    opt.zero_grad()
                    loss.backward()
                    opt.step()

                    bs = xb.size(0)
                    tot += loss.item() * bs
                    cnt += bs

                avg_loss = tot / max(1, cnt)
                ep_mins  = (time.time() - ep_start) / 60.0
                elapsed  = (time.time() - start_time) / 60.0

                history.append({
                    "student_idx": idx,
                    "student_name": name,
                    "epoch": ep,
                    "avg_loss": avg_loss,
                    "epoch_time_min": ep_mins,
                    "elapsed_time_min": elapsed,
                })

                print(f"[BIM-NOV {name}] Epoch {ep:02d} | avg_loss={avg_loss:.4f} | epoch={ep_mins:.2f} min | elapsed={elapsed:.2f} min")

        finally:
            monitor.stop()
            stop_heartbeat(hb_flag, hb_thread)

        hist_path = os.path.join(BIM_NOV_LOGS_DIR, f"{name}_bim_train_history.csv")
        pd.DataFrame(history).to_csv(hist_path, index=False)

        outp = os.path.join(BIM_NOV_STUDENTS_ADV_DIR, f"{name}_bim_adv.pth")
        torch.save(stu.to("cpu").state_dict(), outp)
        adv_paths.append((idx, name, outp))
        print("Saved BIM novel adv student:", outp)

        summary_row = monitor.summary()
        summary_row.update({
            "student": name,
            "attack_train_type": "bim_nov",
            "epochs": EPOCHS_STU,
            "learning_rate": LR_STUDENT,
            "iters": BIM_ITERS,
            "total_elapsed_minutes": (time.time() - start_time) / 60.0,
        })
        append_summary_row(summary_row, BIM_NOV_TRAIN_RESOURCE_CSV)

    return sorted(adv_paths, key=lambda x: x[0])


def adv_train_novel_students_pgd(novel_student_paths):
    adv_paths = []

    print("\n[Novel] Training PGD adv novel students")
    for idx, name, sp in novel_student_paths:
        stu = ElecSeq2SeqTransformer(
            d_feats=D_FEATS,
            d_model=D_MODEL,
            nhead=N_HEAD,
            num_layers=N_LAYERS,
            dim_ff=DIM_FF,
            dropout=DROPOUT,
            hist_steps=HIST_STEPS,
            hzn_steps=HZN_STEPS,
            patch=PATCH
        ).to(device)
        stu.load_state_dict(torch.load(sp, map_location=device))

        opt = optim.AdamW(stu.parameters(), lr=LR_STUDENT)
        history = []

        hb_flag, hb_thread = start_heartbeat(f"{name}_pgd_train")
        monitor = ResourceMonitor(sample_interval=2.0)
        start_time = time.time()
        monitor.start()

        try:
            for ep in range(1, EPOCHS_STU + 1):
                stu.train()
                tot = cnt = 0
                ep_start = time.time()

                for xb, yb in train_dl:
                    xb = xb.to(device, non_blocking=True)
                    yb = yb.to(device, non_blocking=True)

                    pred_c = stu(xb)
                    Lc = rmse_only(pred_c, yb)

                    adv_losses = []
                    for eps in PGD_TRAIN_EPS_LIST:
                        xa = pgd_attack(
                            stu, xb, yb,
                            eps=eps,
                            alpha=eps / PGD_ITERS,
                            iters=PGD_ITERS,
                            random_start=PGD_RANDOM_START_TRAIN
                        )
                        pred_a = stu(xa)
                        La = rmse_only(pred_a, yb)
                        adv_losses.append(La)

                    La_mean = torch.stack(adv_losses).mean() if len(adv_losses) > 0 else 0.0 * Lc
                    loss = 0.25 * Lc + 0.75 * La_mean

                    opt.zero_grad()
                    loss.backward()
                    opt.step()

                    bs = xb.size(0)
                    tot += loss.item() * bs
                    cnt += bs

                avg_loss = tot / max(1, cnt)
                ep_mins  = (time.time() - ep_start) / 60.0
                elapsed  = (time.time() - start_time) / 60.0

                history.append({
                    "student_idx": idx,
                    "student_name": name,
                    "epoch": ep,
                    "avg_loss": avg_loss,
                    "epoch_time_min": ep_mins,
                    "elapsed_time_min": elapsed,
                })

                print(f"[PGD-NOV {name}] Epoch {ep:02d} | avg_loss={avg_loss:.4f} | epoch={ep_mins:.2f} min | elapsed={elapsed:.2f} min")

        finally:
            monitor.stop()
            stop_heartbeat(hb_flag, hb_thread)

        hist_path = os.path.join(PGD_NOV_LOGS_DIR, f"{name}_pgd_train_history.csv")
        pd.DataFrame(history).to_csv(hist_path, index=False)

        outp = os.path.join(PGD_NOV_STUDENTS_ADV_DIR, f"{name}_pgd_adv.pth")
        torch.save(stu.to("cpu").state_dict(), outp)
        adv_paths.append((idx, name, outp))
        print("Saved PGD novel adv student:", outp)

        summary_row = monitor.summary()
        summary_row.update({
            "student": name,
            "attack_train_type": "pgd_nov",
            "epochs": EPOCHS_STU,
            "learning_rate": LR_STUDENT,
            "iters": PGD_ITERS,
            "total_elapsed_minutes": (time.time() - start_time) / 60.0,
        })
        append_summary_row(summary_row, PGD_NOV_TRAIN_RESOURCE_CSV)

    return sorted(adv_paths, key=lambda x: x[0])

In [ ]:
# =========================================================
# SEAMLESS EXECUTION: FGSM -> BIM -> PGD
# =========================================================

nov_fgsm_adv_paths = adv_train_novel_students_fgsm(novel_student_paths)
nov_bim_adv_paths  = adv_train_novel_students_bim(novel_student_paths)
nov_pgd_adv_paths  = adv_train_novel_students_pgd(novel_student_paths)


[Novel] Training FGSM adv novel students


/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[FGSM-NOV student_05_enc_attn] Epoch 01 | avg_loss=0.1199 | epoch=1.04 min | elapsed=1.04 min
[FGSM-NOV student_05_enc_attn] Epoch 02 | avg_loss=0.1051 | epoch=1.04 min | elapsed=2.07 min
[FGSM-NOV student_05_enc_attn] Epoch 03 | avg_loss=0.1018 | epoch=1.03 min | elapsed=3.11 min
[FGSM-NOV student_05_enc_attn] Epoch 04 | avg_loss=0.0996 | epoch=1.04 min | elapsed=4.14 min
[FGSM-NOV student_05_enc_attn] Epoch 05 | avg_loss=0.0975 | epoch=1.04 min | elapsed=5.18 min
Saved FGSM novel adv student: /content/drive/MyDrive/298/Elec/proj_v3/novel/fgsm/students_adv_nov_fgsm/student_05_enc_attn_fgsm_adv.pth


/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))
/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[FGSM-NOV student_06_dec_attn] Epoch 01 | avg_loss=0.1175 | epoch=1.08 min | elapsed=1.08 min
[FGSM-NOV student_06_dec_attn] Epoch 02 | avg_loss=0.1048 | epoch=1.05 min | elapsed=2.13 min
[FGSM-NOV student_06_dec_attn] Epoch 03 | avg_loss=0.1015 | epoch=1.04 min | elapsed=3.17 min
[FGSM-NOV student_06_dec_attn] Epoch 04 | avg_loss=0.0993 | epoch=1.03 min | elapsed=4.20 min
[FGSM-NOV student_06_dec_attn] Epoch 05 | avg_loss=0.0974 | epoch=1.05 min | elapsed=5.25 min
Saved FGSM novel adv student: /content/drive/MyDrive/298/Elec/proj_v3/novel/fgsm/students_adv_nov_fgsm/student_06_dec_attn_fgsm_adv.pth


...

/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))
/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[PGD-NOV student_07_ffn] Epoch 01 | avg_loss=0.1468 | epoch=5.81 min | elapsed=5.81 min
[heartbeat:student_07_ffn_pgd_train] running... elapsed=10.00 min
[PGD-NOV student_07_ffn] Epoch 02 | avg_loss=0.1449 | epoch=5.82 min | elapsed=11.63 min
[PGD-NOV student_07_ffn] Epoch 03 | avg_loss=0.1438 | epoch=5.84 min | elapsed=17.47 min
[heartbeat:student_07_ffn_pgd_train] running... elapsed=20.00 min
[PGD-NOV student_07_ffn] Epoch 04 | avg_loss=0.1421 | epoch=5.79 min | elapsed=23.26 min
[PGD-NOV student_07_ffn] Epoch 05 | avg_loss=0.1409 | epoch=5.82 min | elapsed=29.08 min
Saved PGD novel adv student: /content/drive/MyDrive/298/Elec/proj_v3/novel/pgd/students_adv_nov_pgd/student_07_ffn_pgd_adv.pth


/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))
/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[PGD-NOV student_08_norms] Epoch 01 | avg_loss=0.1464 | epoch=5.75 min | elapsed=5.75 min
[heartbeat:student_08_norms_pgd_train] running... elapsed=10.00 min
[PGD-NOV student_08_norms] Epoch 02 | avg_loss=0.1446 | epoch=5.85 min | elapsed=11.60 min
[PGD-NOV student_08_norms] Epoch 03 | avg_loss=0.1437 | epoch=5.78 min | elapsed=17.38 min
[heartbeat:student_08_norms_pgd_train] running... elapsed=20.00 min
[PGD-NOV student_08_norms] Epoch 04 | avg_loss=0.1434 | epoch=5.82 min | elapsed=23.19 min
[PGD-NOV student_08_norms] Epoch 05 | avg_loss=0.1425 | epoch=5.85 min | elapsed=29.04 min
Saved PGD novel adv student: /content/drive/MyDrive/298/Elec/proj_v3/novel/pgd/students_adv_nov_pgd/student_08_norms_pgd_adv.pth


/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
# =========================================================
# LOAD NOVEL ADV STUDENT POOLS
# =========================================================

def load_novel_student_pool(path_tuples):
    models = []
    names  = []

    for _, name, path in path_tuples:
        m = ElecSeq2SeqTransformer(
            d_feats=D_FEATS,
            d_model=D_MODEL,
            nhead=N_HEAD,
            num_layers=N_LAYERS,
            dim_ff=DIM_FF,
            dropout=DROPOUT,
            hist_steps=HIST_STEPS,
            hzn_steps=HZN_STEPS,
            patch=PATCH
        ).to(device)
        m.load_state_dict(torch.load(path, map_location=device))
        m.eval()
        models.append(m)
        names.append(os.path.basename(path).replace(".pth", ""))

    return models, names

fgsm_nov_students, fgsm_nov_names = load_novel_student_pool(nov_fgsm_adv_paths)
bim_nov_students,  bim_nov_names  = load_novel_student_pool(nov_bim_adv_paths)
pgd_nov_students,  pgd_nov_names  = load_novel_student_pool(nov_pgd_adv_paths)

print("Loaded FGSM novel students:", len(fgsm_nov_students))
print("Loaded BIM novel students :", len(bim_nov_students))
print("Loaded PGD novel students :", len(pgd_nov_students))

/tmp/ipykernel_759/4119019740.py:63: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


Loaded FGSM novel students: 4
Loaded BIM novel students : 4
Loaded PGD novel students : 4


In [ ]:
# =========================================================
# CLEAN EVAL HELPER
# =========================================================

def save_novel_clean_eval(base_model, students, names, out_csv, ensemble_name):
    rows = []

    base_val = eval_one_epoch(base_model, val_dl)
    base_test = eval_one_epoch(base_model, test_dl)

    rows.append({
        "model": "base",
        "val_rmse": base_val,
        "test_rmse": base_test
    })

    for name, stu in zip(names, students):
        val_rmse = eval_one_epoch(stu, val_dl)
        test_rmse = eval_one_epoch(stu, test_dl)
        rows.append({
            "model": name,
            "val_rmse": val_rmse,
            "test_rmse": test_rmse
        })

    ens_clean = eval_ensemble_rmse(students, test_dl)
    rows.append({
        "model": ensemble_name,
        "val_rmse": np.nan,
        "test_rmse": ens_clean
    })

    df = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
    df.to_csv(out_csv, index=False)
    print("Saved clean eval CSV to:", out_csv)
    return df

In [ ]:
# =========================================================
# ATTACK-SPECIFIC EVAL WRAPPERS
# =========================================================

def compute_transfer_matrix_novel(
    students, names, loader, attack_fn, eps_list, extra_kwargs_func,
    out_csv, trace_csv, summary_csv, attack_label="attack"
):
    rows = []

    hb_flag, hb_thread = start_heartbeat(f"{attack_label}_nov_transfer")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_time = time.time()
    monitor.start()

    try:
        for eps in eps_list:
            atk_kwargs = extra_kwargs_func(eps)
            print(f"\n[{attack_label.upper()}-NOV] Transfer matrix eps={eps}")

            for i, atk_name in enumerate(names):
                for j, def_name in enumerate(names):
                    rmse_ij = eval_pair_attack(
                        defender=students[j],
                        attacker=students[i],
                        loader=loader,
                        attack_fn=attack_fn,
                        atk_kwargs=atk_kwargs
                    )
                    rows.append({
                        "epsilon": eps,
                        "attacker": atk_name,
                        "defender": def_name,
                        "rmse_adv": rmse_ij,
                    })
                    print(f"  atk={atk_name} -> def={def_name} | RMSE={rmse_ij:.5f}")
    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)

    save_monitor_outputs(
        monitor,
        trace_csv=trace_csv,
        summary_csv=summary_csv,
        extra_summary={
            "stage": f"{attack_label}_nov_transfer_matrix",
            "total_elapsed_minutes": (time.time() - start_time) / 60.0,
        }
    )

    return df


def run_novel_single_pass_eval(
    attack_name, attack_fn, eps_list, students,
    results_csv, stats_csv, extra_kwargs_func,
    trace_csv, summary_csv, ensemble_label
):
    rows = []

    hb_flag, hb_thread = start_heartbeat(f"{attack_name}_nov_single_pass")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_time = time.time()
    monitor.start()

    try:
        base_clean = eval_clean_rmse(best_base, test_dl)
        nov_clean = eval_ensemble_rmse(students, test_dl)

        rows.append(["clean", None, "base", base_clean, 1, SEED])
        rows.append(["clean", None, ensemble_label, nov_clean, 1, SEED])

        print(f"[{attack_name.upper()}-NOV] clean | base={base_clean:.5f} | {ensemble_label}={nov_clean:.5f}")

        for eps in eps_list:
            atk_kwargs = extra_kwargs_func(eps)

            base_adv = eval_pair_attack(
                defender=best_base,
                attacker=best_base,
                loader=test_dl,
                attack_fn=attack_fn,
                atk_kwargs=atk_kwargs
            )

            def atk_kwargs_wrapper():
                return atk_kwargs

            nov_rand = eval_random_switch_attack(
                models=students,
                loader=test_dl,
                attack_fn=attack_fn,
                atk_kwargs_func=atk_kwargs_wrapper
            )

            rows.append([attack_name, eps, "base", base_adv, 1, SEED])
            rows.append([attack_name, eps, f"{attack_name}_nov_rand", nov_rand, 1, SEED])

            print(f"[{attack_name.upper()}-NOV] eps={eps:.2f} | base={base_adv:.5f} | rand={nov_rand:.5f}")

    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    df_runs = pd.DataFrame(rows, columns=["attack", "epsilon", "model", "RMSE", "run", "seed"])
    df_runs.to_csv(results_csv, index=False)

    df_stats = (
        df_runs
        .groupby(["attack", "epsilon", "model"])["RMSE"]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
        .sort_values(["attack", "epsilon", "model"])
    )
    df_stats.to_csv(stats_csv, index=False)

    save_monitor_outputs(
        monitor,
        trace_csv=trace_csv,
        summary_csv=summary_csv,
        extra_summary={
            "stage": f"{attack_name}_nov_single_pass_eval",
            "total_elapsed_minutes": (time.time() - start_time) / 60.0,
        }
    )

    return df_runs, df_stats

In [ ]:
# =========================================================
# FGSM NOVEL EVAL
# =========================================================

def fgsm_nov_kwargs(eps):
    return {"eps": eps, "alpha": eps * RFGSM_ALPHA_FACTOR}

_ = save_novel_clean_eval(best_base, fgsm_nov_students, fgsm_nov_names, NOV_FGSM_CLEAN_EVAL_CSV, "nov_fgsm_ensemble")
_ = compute_weight_diversity(fgsm_nov_students, fgsm_nov_names, NOV_FGSM_WEIGHT_CSV)
_ = compute_prediction_diversity(fgsm_nov_students, test_dl, NOV_FGSM_PRED_CSV)

_ = compute_transfer_matrix_novel(
    students=fgsm_nov_students,
    names=fgsm_nov_names,
    loader=test_dl,
    attack_fn=rfgsm_attack,
    eps_list=FGSM_EVAL_EPS_LIST,
    extra_kwargs_func=fgsm_nov_kwargs,
    out_csv=NOV_FGSM_XFER_CSV,
    trace_csv=FGSM_NOV_XFER_TRACE_CSV,
    summary_csv=FGSM_NOV_XFER_SUMMARY_CSV,
    attack_label="fgsm",
)

df_fgsm_nov_runs, df_fgsm_nov_stats = run_novel_single_pass_eval(
    attack_name="fgsm",
    attack_fn=rfgsm_attack,
    eps_list=FGSM_EVAL_EPS_LIST,
    students=fgsm_nov_students,
    results_csv=NOV_FGSM_RUNS_CSV,
    stats_csv=NOV_FGSM_STATS_CSV,
    extra_kwargs_func=fgsm_nov_kwargs,
    trace_csv=FGSM_NOV_EVAL_TRACE_CSV,
    summary_csv=FGSM_NOV_EVAL_SUMMARY_CSV,
    ensemble_label="nov_fgsm_ensemble",
)

Saved clean eval CSV to: /content/drive/MyDrive/298/Elec/proj_v3/novel/fgsm/results/nov_fgsm_clean_eval.csv
Saved weight diversity to: /content/drive/MyDrive/298/Elec/proj_v3/novel/fgsm/results/nov_fgsm_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Elec/proj_v3/novel/fgsm/results/nov_fgsm_prediction_diversity.csv

[FGSM-NOV] Transfer matrix eps=0.1
  atk=student_05_enc_attn_fgsm_adv -> def=student_05_enc_attn_fgsm_adv | RMSE=0.15536
  atk=student_05_enc_attn_fgsm_adv -> def=student_06_dec_attn_fgsm_adv | RMSE=0.12213
  atk=student_05_enc_attn_fgsm_adv -> def=student_07_ffn_fgsm_adv | RMSE=0.12484
  atk=student_05_enc_attn_fgsm_adv -> def=student_08_norms_fgsm_adv | RMSE=0.11832
  atk=student_06_dec_attn_fgsm_adv -> def=student_05_enc_attn_fgsm_adv | RMSE=0.14392
  atk=student_06_dec_attn_fgsm_adv -> def=student_06_dec_attn_fgsm_adv | RMSE=0.12648
  atk=student_06_dec_attn_fgsm_adv -> def=student_07_ffn_fgsm_adv | RMSE=0.12707
  atk=student_06_dec_attn_f

/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


[FGSM-NOV] clean | base=0.11526 | nov_fgsm_ensemble=0.11416
[FGSM-NOV] eps=0.10 | base=0.17681 | rand=0.13267
[FGSM-NOV] eps=0.20 | base=0.24134 | rand=0.12364
[FGSM-NOV] eps=0.30 | base=0.27607 | rand=0.11602
[FGSM-NOV] eps=0.50 | base=0.31682 | rand=0.11691
Saved resource trace to: /content/drive/MyDrive/298/Elec/proj_v3/novel/computational_metrics/fgsm_nov_eval_resource_trace.csv
Saved resource summary to: /content/drive/MyDrive/298/Elec/proj_v3/novel/computational_metrics/fgsm_nov_eval_resource_summary.csv
Resource summary: {'n_samples': 12, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 1932.484375, 'max_gpu_mem_mb': 1637.0, 'avg_gpu_util_percent': 31.416666666666668, 'gpu_active_percent_of_samples': 91.66666666666666, 'est_gpu_energy_j': 3625.9033264863488, 'stage': 'fgsm_nov_single_pass_eval', 'total_elapsed_minutes': 0.42047679821650186}


/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
# =========================================================
# BIM NOVEL EVAL
# =========================================================

def bim_nov_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps / BIM_ITERS,
        "iters": BIM_ITERS,
        "random_start": BIM_RANDOM_START_EVAL,
    }

_ = save_novel_clean_eval(best_base, bim_nov_students, bim_nov_names, NOV_BIM_CLEAN_EVAL_CSV, "nov_bim_ensemble")
_ = compute_weight_diversity(bim_nov_students, bim_nov_names, NOV_BIM_WEIGHT_CSV)
_ = compute_prediction_diversity(bim_nov_students, test_dl, NOV_BIM_PRED_CSV)

_ = compute_transfer_matrix_novel(
    students=bim_nov_students,
    names=bim_nov_names,
    loader=test_dl,
    attack_fn=bim_attack,
    eps_list=BIM_EVAL_EPS_LIST,
    extra_kwargs_func=bim_nov_kwargs,
    out_csv=NOV_BIM_XFER_CSV,
    trace_csv=BIM_NOV_XFER_TRACE_CSV,
    summary_csv=BIM_NOV_XFER_SUMMARY_CSV,
    attack_label="bim",
)

df_bim_nov_runs, df_bim_nov_stats = run_novel_single_pass_eval(
    attack_name="bim",
    attack_fn=bim_attack,
    eps_list=BIM_EVAL_EPS_LIST,
    students=bim_nov_students,
    results_csv=NOV_BIM_RUNS_CSV,
    stats_csv=NOV_BIM_STATS_CSV,
    extra_kwargs_func=bim_nov_kwargs,
    trace_csv=BIM_NOV_EVAL_TRACE_CSV,
    summary_csv=BIM_NOV_EVAL_SUMMARY_CSV,
    ensemble_label="nov_bim_ensemble",
)

Saved clean eval CSV to: /content/drive/MyDrive/298/Elec/proj_v3/novel/bim/results/nov_bim_clean_eval.csv
Saved weight diversity to: /content/drive/MyDrive/298/Elec/proj_v3/novel/bim/results/nov_bim_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Elec/proj_v3/novel/bim/results/nov_bim_prediction_diversity.csv

[BIM-NOV] Transfer matrix eps=0.1
  atk=student_05_enc_attn_bim_adv -> def=student_05_enc_attn_bim_adv | RMSE=0.22373
  atk=student_05_enc_attn_bim_adv -> def=student_06_dec_attn_bim_adv | RMSE=0.23997
  atk=student_05_enc_attn_bim_adv -> def=student_07_ffn_bim_adv | RMSE=0.23811
  atk=student_05_enc_attn_bim_adv -> def=student_08_norms_bim_adv | RMSE=0.24413
  atk=student_06_dec_attn_bim_adv -> def=student_05_enc_attn_bim_adv | RMSE=0.22372
  atk=student_06_dec_attn_bim_adv -> def=student_06_dec_attn_bim_adv | RMSE=0.23999
  atk=student_06_dec_attn_bim_adv -> def=student_07_ffn_bim_adv | RMSE=0.23812
  atk=student_06_dec_attn_bim_adv -> def=student

/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


[BIM-NOV] clean | base=0.11526 | nov_bim_ensemble=0.23602
[BIM-NOV] eps=0.10 | base=0.16998 | rand=0.23642
[BIM-NOV] eps=0.20 | base=0.25706 | rand=0.23509
[BIM-NOV] eps=0.30 | base=0.32253 | rand=0.23500
[BIM-NOV] eps=0.50 | base=0.38504 | rand=0.23664
Saved resource trace to: /content/drive/MyDrive/298/Elec/proj_v3/novel/computational_metrics/bim_nov_eval_resource_trace.csv
Saved resource summary to: /content/drive/MyDrive/298/Elec/proj_v3/novel/computational_metrics/bim_nov_eval_resource_summary.csv
Resource summary: {'n_samples': 93, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 1932.80859375, 'max_gpu_mem_mb': 1715.0, 'avg_gpu_util_percent': 46.72043010752688, 'gpu_active_percent_of_samples': 98.9247311827957, 'est_gpu_energy_j': 31417.409149930474, 'stage': 'bim_nov_single_pass_eval', 'total_elapsed_minutes': 3.1432026267051696}


/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
# =========================================================
# PGD NOVEL EVAL
# =========================================================

def pgd_nov_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps / PGD_ITERS,
        "iters": PGD_ITERS,
        "random_start": PGD_RANDOM_START_EVAL,
    }

_ = save_novel_clean_eval(best_base, pgd_nov_students, pgd_nov_names, NOV_PGD_CLEAN_EVAL_CSV, "nov_pgd_ensemble")
_ = compute_weight_diversity(pgd_nov_students, pgd_nov_names, NOV_PGD_WEIGHT_CSV)
_ = compute_prediction_diversity(pgd_nov_students, test_dl, NOV_PGD_PRED_CSV)

_ = compute_transfer_matrix_novel(
    students=pgd_nov_students,
    names=pgd_nov_names,
    loader=test_dl,
    attack_fn=pgd_attack,
    eps_list=PGD_EVAL_EPS_LIST,
    extra_kwargs_func=pgd_nov_kwargs,
    out_csv=NOV_PGD_XFER_CSV,
    trace_csv=PGD_NOV_XFER_TRACE_CSV,
    summary_csv=PGD_NOV_XFER_SUMMARY_CSV,
    attack_label="pgd",
)

df_pgd_nov_runs, df_pgd_nov_stats = run_novel_single_pass_eval(
    attack_name="pgd",
    attack_fn=pgd_attack,
    eps_list=PGD_EVAL_EPS_LIST,
    students=pgd_nov_students,
    results_csv=NOV_PGD_RUNS_CSV,
    stats_csv=NOV_PGD_STATS_CSV,
    extra_kwargs_func=pgd_nov_kwargs,
    trace_csv=PGD_NOV_EVAL_TRACE_CSV,
    summary_csv=PGD_NOV_EVAL_SUMMARY_CSV,
    ensemble_label="nov_pgd_ensemble",
)

Saved clean eval CSV to: /content/drive/MyDrive/298/Elec/proj_v3/novel/pgd/results/nov_pgd_clean_eval.csv
Saved weight diversity to: /content/drive/MyDrive/298/Elec/proj_v3/novel/pgd/results/nov_pgd_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Elec/proj_v3/novel/pgd/results/nov_pgd_prediction_diversity.csv

[PGD-NOV] Transfer matrix eps=0.1
  atk=student_05_enc_attn_pgd_adv -> def=student_05_enc_attn_pgd_adv | RMSE=0.21735
  atk=student_05_enc_attn_pgd_adv -> def=student_06_dec_attn_pgd_adv | RMSE=0.21729
  atk=student_05_enc_attn_pgd_adv -> def=student_07_ffn_pgd_adv | RMSE=0.21314
  atk=student_05_enc_attn_pgd_adv -> def=student_08_norms_pgd_adv | RMSE=0.19321
  atk=student_06_dec_attn_pgd_adv -> def=student_05_enc_attn_pgd_adv | RMSE=0.21710
  atk=student_06_dec_attn_pgd_adv -> def=student_06_dec_attn_pgd_adv | RMSE=0.21773
  atk=student_06_dec_attn_pgd_adv -> def=student_07_ffn_pgd_adv | RMSE=0.21302
  atk=student_06_dec_attn_pgd_adv -> def=student

/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


[PGD-NOV] clean | base=0.11526 | nov_pgd_ensemble=0.20892
[PGD-NOV] eps=0.10 | base=0.16992 | rand=0.21035
[PGD-NOV] eps=0.20 | base=0.25764 | rand=0.21169
[PGD-NOV] eps=0.30 | base=0.32280 | rand=0.21774
[PGD-NOV] eps=0.50 | base=0.38507 | rand=0.22905
Saved resource trace to: /content/drive/MyDrive/298/Elec/proj_v3/novel/computational_metrics/pgd_nov_eval_resource_trace.csv
Saved resource summary to: /content/drive/MyDrive/298/Elec/proj_v3/novel/computational_metrics/pgd_nov_eval_resource_summary.csv
Resource summary: {'n_samples': 93, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 1932.9921875, 'max_gpu_mem_mb': 1777.0, 'avg_gpu_util_percent': 46.43010752688172, 'gpu_active_percent_of_samples': 98.9247311827957, 'est_gpu_energy_j': 31539.566150733233, 'stage': 'pgd_nov_single_pass_eval', 'total_elapsed_minutes': 3.143471844991048}


/tmp/ipykernel_759/498116179.py:108: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
print("\n================ ECL NOVEL PROJ_V3 OUTPUTS ================\n")

print("Novel model sizes           :", NOVEL_MODEL_SIZES_CSV)

print("\nFGSM NOV clean eval         :", NOV_FGSM_CLEAN_EVAL_CSV)
print("FGSM NOV train resource     :", FGSM_NOV_TRAIN_RESOURCE_CSV)
print("FGSM NOV weight diversity   :", NOV_FGSM_WEIGHT_CSV)
print("FGSM NOV prediction diversity:", NOV_FGSM_PRED_CSV)
print("FGSM NOV transfer matrix    :", NOV_FGSM_XFER_CSV)
print("FGSM NOV transfer trace     :", FGSM_NOV_XFER_TRACE_CSV)
print("FGSM NOV transfer summary   :", FGSM_NOV_XFER_SUMMARY_CSV)
print("FGSM NOV single-pass eval   :", NOV_FGSM_RUNS_CSV)
print("FGSM NOV single-pass stats  :", NOV_FGSM_STATS_CSV)
print("FGSM NOV eval trace         :", FGSM_NOV_EVAL_TRACE_CSV)
print("FGSM NOV eval summary       :", FGSM_NOV_EVAL_SUMMARY_CSV)

print("\nBIM NOV clean eval          :", NOV_BIM_CLEAN_EVAL_CSV)
print("BIM NOV train resource      :", BIM_NOV_TRAIN_RESOURCE_CSV)
print("BIM NOV weight diversity    :", NOV_BIM_WEIGHT_CSV)
print("BIM NOV prediction diversity:", NOV_BIM_PRED_CSV)
print("BIM NOV transfer matrix     :", NOV_BIM_XFER_CSV)
print("BIM NOV transfer trace      :", BIM_NOV_XFER_TRACE_CSV)
print("BIM NOV transfer summary    :", BIM_NOV_XFER_SUMMARY_CSV)
print("BIM NOV single-pass eval    :", NOV_BIM_RUNS_CSV)
print("BIM NOV single-pass stats   :", NOV_BIM_STATS_CSV)
print("BIM NOV eval trace          :", BIM_NOV_EVAL_TRACE_CSV)
print("BIM NOV eval summary        :", BIM_NOV_EVAL_SUMMARY_CSV)

print("\nPGD NOV clean eval          :", NOV_PGD_CLEAN_EVAL_CSV)
print("PGD NOV train resource      :", PGD_NOV_TRAIN_RESOURCE_CSV)
print("PGD NOV weight diversity    :", NOV_PGD_WEIGHT_CSV)
print("PGD NOV prediction diversity:", NOV_PGD_PRED_CSV)
print("PGD NOV transfer matrix     :", NOV_PGD_XFER_CSV)
print("PGD NOV transfer trace      :", PGD_NOV_XFER_TRACE_CSV)
print("PGD NOV transfer summary    :", PGD_NOV_XFER_SUMMARY_CSV)
print("PGD NOV single-pass eval    :", NOV_PGD_RUNS_CSV)
print("PGD NOV single-pass stats   :", NOV_PGD_STATS_CSV)
print("PGD NOV eval trace          :", PGD_NOV_EVAL_TRACE_CSV)
print("PGD NOV eval summary        :", PGD_NOV_EVAL_SUMMARY_CSV)


================ ECL NOVEL PROJ_V3 OUTPUTS ================

Novel model sizes           : /content/drive/MyDrive/298/Elec/proj_v3/novel/results/model_sizes_nov.csv

FGSM NOV clean eval         : /content/drive/MyDrive/298/Elec/proj_v3/novel/fgsm/results/nov_fgsm_clean_eval.csv
FGSM NOV train resource     : /content/drive/MyDrive/298/Elec/proj_v3/novel/computational_metrics/fgsm_nov_student_train_resource_summary.csv
FGSM NOV weight diversity   : /content/drive/MyDrive/298/Elec/proj_v3/novel/fgsm/results/nov_fgsm_weight_diversity.csv
FGSM NOV prediction diversity: /content/drive/MyDrive/298/Elec/proj_v3/novel/fgsm/results/nov_fgsm_prediction_diversity.csv
FGSM NOV transfer matrix    : /content/drive/MyDrive/298/Elec/proj_v3/novel/fgsm/results/nov_fgsm_transfer_matrix.csv
FGSM NOV transfer trace     : /content/drive/MyDrive/298/Elec/proj_v3/novel/computational_metrics/fgsm_nov_transfer_resource_trace.csv
FGSM NOV transfer summary   : /content/drive/MyDrive/298/Elec/proj_v3/novel/comput

In [ ]:
# FINAL COLAB CLEANUP CELL
# Put this as the very last cell in the notebook.

import gc
import time
import torch
from google.colab import runtime

print("Final cleanup starting...")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(10)

print("Disconnecting and deleting runtime now...")
runtime.unassign()

Final cleanup starting...
Disconnecting and deleting runtime now...


In [ ]:
# =========================================================
# EXPORT ECL PROJ_V3 COMPUTATIONAL METRICS TO ONE EXCEL FILE
# Verified against your actual ECL proj_v3 output paths
# =========================================================

# -----------------------------
# STEP 0: Mount Google Drive
# -----------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    print("⚠️ Not running in Colab? Make sure paths are accessible.")

import os
import re
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)

# -----------------------------
# base paths
# -----------------------------
BASE_PROJ_DIR = "/content/drive/MyDrive/298/Elec/proj_v3"
COMP_METRICS_DIR = os.path.join(BASE_PROJ_DIR, "computational_metrics")

NOVEL_DIR = os.path.join(BASE_PROJ_DIR, "novel")
NOVEL_COMP_DIR = os.path.join(NOVEL_DIR, "computational_metrics")
NOVEL_RESULTS_DIR = os.path.join(NOVEL_DIR, "results")

OUTPUT_XLSX = os.path.join(BASE_PROJ_DIR, "ecl_computational_metrics_report.xlsx")

if not os.path.exists(BASE_PROJ_DIR):
    raise FileNotFoundError(
        f"❌ Base directory not found:\n{BASE_PROJ_DIR}\n"
        "Make sure Google Drive is mounted and the path is correct."
    )

print("✅ Drive mounted and base directory found.")
print("Base project dir:", BASE_PROJ_DIR)

# -----------------------------
# helper functions
# -----------------------------
def safe_read_csv(path):
    if os.path.exists(path):
        try:
            return pd.read_csv(path)
        except Exception as e:
            print(f"⚠️ Could not read {path}: {e}")
            return None
    else:
        print(f"⚠️ Missing file: {path}")
        return None

def add_source_columns(df, section, metric_name, source_path):
    if df is None or len(df) == 0:
        return None
    df = df.copy()
    df.insert(0, "section", section)
    df.insert(1, "metric_name", metric_name)
    df.insert(2, "source_csv", source_path)
    return df

def full_csv(path, section, metric_name):
    df = safe_read_csv(path)
    if df is None or len(df) == 0:
        return None
    return add_source_columns(df, section, metric_name, path)

def safe_sheet_name(name, used_names):
    cleaned = re.sub(r'[\[\]\:\*\?\/\\]', '_', name)
    cleaned = cleaned[:31]
    original = cleaned
    i = 1
    while cleaned in used_names:
        suffix = f"_{i}"
        cleaned = original[:31 - len(suffix)] + suffix
        i += 1
    used_names.add(cleaned)
    return cleaned

def auto_adjust_column_widths_openpyxl(ws, df):
    for col_idx, col_name in enumerate(df.columns, start=1):
        max_len = len(str(col_name))
        if len(df) > 0:
            for val in df[col_name].astype(str).fillna(""):
                max_len = max(max_len, len(val))
        max_len = min(max_len + 2, 40)
        ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = max_len

# -----------------------------
# verified file definitions
# -----------------------------
files_summary = [
    # Base
    ("Base", "Base training summary",
     os.path.join(COMP_METRICS_DIR, "base_resource_summary.csv")),

    ("Base", "Base model profile",
     os.path.join(COMP_METRICS_DIR, "base_model_profile.csv")),

    # Original / vanilla students
    ("Original", "FGSM student train summary",
     os.path.join(COMP_METRICS_DIR, "fgsm_student_train_resource_summary.csv")),
    ("Original", "BIM student train summary",
     os.path.join(COMP_METRICS_DIR, "bim_student_train_resource_summary.csv")),
    ("Original", "PGD student train summary",
     os.path.join(COMP_METRICS_DIR, "pgd_student_train_resource_summary.csv")),

    ("Original", "FGSM transfer summary",
     os.path.join(COMP_METRICS_DIR, "fgsm_transfer_resource_summary.csv")),
    ("Original", "BIM transfer summary",
     os.path.join(COMP_METRICS_DIR, "bim_transfer_resource_summary.csv")),
    ("Original", "PGD transfer summary",
     os.path.join(COMP_METRICS_DIR, "pgd_transfer_resource_summary.csv")),

    ("Original", "FGSM eval summary",
     os.path.join(COMP_METRICS_DIR, "fgsm_eval_resource_summary.csv")),
    ("Original", "BIM eval summary",
     os.path.join(COMP_METRICS_DIR, "bim_eval_resource_summary.csv")),
    ("Original", "PGD eval summary",
     os.path.join(COMP_METRICS_DIR, "pgd_eval_resource_summary.csv")),

    # Novel students
    ("Novel", "Model sizes",
     os.path.join(NOVEL_RESULTS_DIR, "model_sizes_nov.csv")),

    ("Novel", "FGSM NOV student train summary",
     os.path.join(NOVEL_COMP_DIR, "fgsm_nov_student_train_resource_summary.csv")),
    ("Novel", "BIM NOV student train summary",
     os.path.join(NOVEL_COMP_DIR, "bim_nov_student_train_resource_summary.csv")),
    ("Novel", "PGD NOV student train summary",
     os.path.join(NOVEL_COMP_DIR, "pgd_nov_student_train_resource_summary.csv")),

    ("Novel", "FGSM NOV transfer summary",
     os.path.join(NOVEL_COMP_DIR, "fgsm_nov_transfer_resource_summary.csv")),
    ("Novel", "BIM NOV transfer summary",
     os.path.join(NOVEL_COMP_DIR, "bim_nov_transfer_resource_summary.csv")),
    ("Novel", "PGD NOV transfer summary",
     os.path.join(NOVEL_COMP_DIR, "pgd_nov_transfer_resource_summary.csv")),

    ("Novel", "FGSM NOV eval summary",
     os.path.join(NOVEL_COMP_DIR, "fgsm_nov_eval_resource_summary.csv")),
    ("Novel", "BIM NOV eval summary",
     os.path.join(NOVEL_COMP_DIR, "bim_nov_eval_resource_summary.csv")),
    ("Novel", "PGD NOV eval summary",
     os.path.join(NOVEL_COMP_DIR, "pgd_nov_eval_resource_summary.csv")),
]

# -----------------------------
# load all dataframes
# -----------------------------
loaded = {}
all_tables = []

for section, metric_name, path in files_summary:
    df = full_csv(path, section, metric_name)
    if df is not None:
        loaded[(section, metric_name)] = df
        all_tables.append(df)

combined = pd.concat(all_tables, ignore_index=True) if all_tables else pd.DataFrame()

if len(combined) == 0:
    raise ValueError("❌ No ECL computational metrics CSVs were loaded.")

# -----------------------------
# grouped views
# -----------------------------
base_model_df = combined[
    combined["metric_name"].str.contains(
        "Base training summary|Base model profile|Model sizes",
        case=False,
        na=False
    )
].copy()

training_df = combined[
    combined["metric_name"].str.contains("train summary", case=False, na=False)
].copy()

transfer_df = combined[
    combined["metric_name"].str.contains("transfer summary", case=False, na=False)
].copy()

evaluation_df = combined[
    combined["metric_name"].str.contains("eval summary", case=False, na=False)
].copy()

original_df = combined[
    combined["section"] == "Original"
].copy()

novel_df = combined[
    combined["section"] == "Novel"
].copy()

compact_cols = [
    "section",
    "metric_name",
    "student",
    "attack_train_type",
    "stage",
    "epochs",
    "learning_rate",
    "iters",
    "n_samples",
    "sample_interval_sec",
    "max_cpu_ram_mb",
    "max_gpu_mem_mb",
    "avg_gpu_util_percent",
    "gpu_active_percent_of_samples",
    "est_gpu_energy_j",
    "total_elapsed_minutes",
    "estimated_macs",
    "estimated_flops",
    "parameter_count",
]
compact_cols = [c for c in compact_cols if c in combined.columns]
compact_df = combined[compact_cols].copy()

# Attack-wise slices
fgsm_df = combined[
    combined["metric_name"].str.contains("FGSM", case=False, na=False)
].copy()

bim_df = combined[
    combined["metric_name"].str.contains("BIM", case=False, na=False)
].copy()

pgd_df = combined[
    combined["metric_name"].str.contains("PGD", case=False, na=False)
].copy()

# -----------------------------
# export to Excel
# -----------------------------
used_names = set()

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    summary_sheets = {
        "All_Combined": combined,
        "Compact_View": compact_df,
        "Base_and_Model": base_model_df,
        "Training": training_df,
        "Transfer": transfer_df,
        "Evaluation": evaluation_df,
        "Original_Only": original_df,
        "Novel_Only": novel_df,
        "FGSM_Only": fgsm_df,
        "BIM_Only": bim_df,
        "PGD_Only": pgd_df,
    }

    for sheet_name, df in summary_sheets.items():
        if df is not None and len(df) > 0:
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            ws = writer.sheets[sheet_name]
            ws.freeze_panes = "A2"
            auto_adjust_column_widths_openpyxl(ws, df)

    for (section, metric_name), df in loaded.items():
        sheet_name = safe_sheet_name(f"{section}_{metric_name}", used_names)
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.sheets[sheet_name]
        ws.freeze_panes = "A2"
        auto_adjust_column_widths_openpyxl(ws, df)

print(f"✅ Excel workbook saved to:\n{OUTPUT_XLSX}")

# -----------------------------
# notebook preview
# -----------------------------
print("\nPreview: All Combined")
display(combined.head())

print("\nPreview: Compact View")
display(compact_df.head())

print("\nLoaded metric files:")
for section, metric_name, path in files_summary:
    exists = os.path.exists(path)
    print(f"[{'OK' if exists else 'MISS'}] {section:8s} | {metric_name:32s} | {path}")

Mounted at /content/drive
✅ Drive mounted and base directory found.
Base project dir: /content/drive/MyDrive/298/Elec/proj_v3
✅ Excel workbook saved to:
/content/drive/MyDrive/298/Elec/proj_v3/ecl_computational_metrics_report.xlsx

Preview: All Combined


,section,metric_name,source_csv,n_samples,sample_interval_sec,max_cpu_ram_mb,max_gpu_mem_mb,avg_gpu_util_percent,gpu_active_percent_of_samples,est_gpu_energy_j,stage,epochs,learning_rate,total_elapsed_minutes,model,estimated_macs,estimated_flops,parameter_count,student,attack_train_type,iters,student_idx,model_name,path,params,size_mb
0,Base,Base training summary,/content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/base_resource_summary.csv,149.0,2.0,1662.167969,893.0,41.020134,99.328859,49261.286399,base_training,30.0,0.0005,5.026334,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Base,Base model profile,/content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/base_model_profile.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,elec_base_transformer,95213568.0,190427136.0,1585921.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Original,FGSM student train summary,/content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/fgsm_student_train_resource_summary.csv,156.0,2.0,1692.781250,1283.0,43.820513,99.358974,54262.151337,NaN,5.0,0.0001,5.262403,NaN,NaN,NaN,NaN,student_01,fgsm,NaN,NaN,NaN,NaN,NaN,NaN
3,Original,FGSM student train summary,/content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/fgsm_student_train_resource_summary.csv,124.0,2.0,1743.019531,1285.0,43.887097,100.000000,51620.739293,NaN,5.0,0.0001,5.249874,NaN,NaN,NaN,NaN,student_02,fgsm,NaN,NaN,NaN,NaN,NaN,NaN
4,Original,FGSM student train summary,/content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/fgsm_student_train_resource_summary.csv,155.0,2.0,1743.679688,1285.0,43.683871,98.709677,54018.772021,NaN,5.0,0.0001,5.228584,NaN,NaN,NaN,NaN,student_03,fgsm,NaN,NaN,NaN,NaN,NaN,NaN



Preview: Compact View


,section,metric_name,student,attack_train_type,stage,epochs,learning_rate,iters,n_samples,sample_interval_sec,max_cpu_ram_mb,max_gpu_mem_mb,avg_gpu_util_percent,gpu_active_percent_of_samples,est_gpu_energy_j,total_elapsed_minutes,estimated_macs,estimated_flops,parameter_count
0,Base,Base training summary,NaN,NaN,base_training,30.0,0.0005,NaN,149.0,2.0,1662.167969,893.0,41.020134,99.328859,49261.286399,5.026334,NaN,NaN,NaN
1,Base,Base model profile,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,95213568.0,190427136.0,1585921.0
2,Original,FGSM student train summary,student_01,fgsm,NaN,5.0,0.0001,NaN,156.0,2.0,1692.781250,1283.0,43.820513,99.358974,54262.151337,5.262403,NaN,NaN,NaN
3,Original,FGSM student train summary,student_02,fgsm,NaN,5.0,0.0001,NaN,124.0,2.0,1743.019531,1285.0,43.887097,100.000000,51620.739293,5.249874,NaN,NaN,NaN
4,Original,FGSM student train summary,student_03,fgsm,NaN,5.0,0.0001,NaN,155.0,2.0,1743.679688,1285.0,43.683871,98.709677,54018.772021,5.228584,NaN,NaN,NaN



Loaded metric files:
[OK] Base     | Base training summary            | /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/base_resource_summary.csv
[OK] Base     | Base model profile               | /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/base_model_profile.csv
[OK] Original | FGSM student train summary       | /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/fgsm_student_train_resource_summary.csv
[OK] Original | BIM student train summary        | /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/bim_student_train_resource_summary.csv
[OK] Original | PGD student train summary        | /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/pgd_student_train_resource_summary.csv
[OK] Original | FGSM transfer summary            | /content/drive/MyDrive/298/Elec/proj_v3/computational_metrics/fgsm_transfer_resource_summary.csv
[OK] Original | BIM transfer summary             | /content/drive/MyDrive/298/Elec/proj_v3/computa